<div style="background: linear-gradient(120deg, #0b1f4b 0%, #1d4f9c 45%, #2a78d6 70%, #1baf7a 100%); padding: 30px 32px 26px 32px; border-radius: 14px; color: #ffffff;">
<div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; opacity: 0.85;">Google Quantum AI &middot; Verifiable Quantum Circuit Compilation</div>
<div style="font-size: 34px; font-weight: 800; line-height: 1.15; margin-top: 8px;">The QCC Visual Guide</div>
<div style="font-size: 17px; margin-top: 6px; opacity: 0.95;">Three tasks, one score, three verifiers &mdash; the system explained in pictures, with an exact simulator, cost anatomy, the public lineage and our negative results</div>
<div style="margin-top: 16px;">
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">invsqrt</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">DQI</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">chemistry</span>
<span style="background: #ffd166; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">portfolio GM 25.4495</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">all figures generated in this notebook</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">official verifiers</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px;">Apache-2.0 code</span>
</div>
</div>

**What this notebook is.** A guide to the competition for newcomers and a lab notebook for experts. It explains the
three circuits, how the score and the three verifiers work, what one qubit or one Toffoli is worth in each task, where
the cost of the best circuits goes, how the public frontier was built and by whom, and which ideas do *not* work, with
the measurements. It ships `kmsim`, a ~150-line numpy simulator of the kickmix subset these circuits use, which
reproduces the official invsqrt verifier exactly and lets you stop a circuit after any gate.

**It also submits.** The `submission.zip` written here is copied byte for byte from our solution notebook (input of
this notebook), so this notebook carries the same public score. Every figure and printed number about the portfolio
is read from that notebook's official-verifier receipt at run time; numbers written in the text describe the portfolio
at the time of writing (2026-11-12): invsqrt `9c708f7b`, DQI `4c81b726` (maoxin123's 392k-Toffoli decoder, 0 failures at
three salts) and the 4,089-Toffoli chemistry loaders, geometric mean 25.4495 against 25.40767 for the previous public
best. The public circuits come from our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas)
dataset and manwithacat's public-circuits dataset.

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Contents</b><br>
<a href="#ack">0. Acknowledgements and who did what</a><br>
<a href="#s1">1. The competition in five minutes</a> &mdash; tasks, score, verifiers, the kickmix instruction set, the one trick everyone uses<br>
<a href="#s2">2. The score, the cost anatomy and the exchange rates</a><br>
<a href="#s3">3. <code>kmsim</code>: an exact numpy simulator for these circuits</a><br>
<a href="#s4">4. invsqrt deep dive</a> &mdash; Toffolis along the circuit, the two phase streams, the error budget, splicing, the landscape<br>
<a href="#s5">5. DQI deep dive</a> &mdash; the readout-window identity, region profiles, liveness, rare failures and board luck<br>
<a href="#s6">6. Chemistry deep dive</a> &mdash; the cost formula, the loaders, the factorization lever<br>
<a href="#s7">7. The public lineage</a> &mdash; how the frontier was built, and by whom<br>
<a href="#s8">8. What does not work, and open problems</a><br>
<a href="#s9">9. Full credits and licences</a>
</div>

If this guide is useful to you, an upvote helps others find it. Corrections and ideas are very welcome in the comments.

<a id="ack"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">0 &middot; Acknowledgements and who did what</div>

Almost every circuit discussed here is other people's public work. Thank you to everyone whose notebooks, datasets and
write-ups this guide stands on. In particular: **Alex Chilton**, whose *Verified Three-Task Portfolio* set the
standard for pinning and re-verifying the public frontier; **maoxin123** for the 392k-Toffoli DQI decoder of our portfolio;
**Areeb Ali** for product retention, the previous public best; **Atakan Aldemir** for the earlier DQI files and the
readout windows, shared products and GF(64) characters behind them; **WCao24** for the invsqrt normaliser;
**Thomas Gamet** for the early-prefix invsqrt circuit; **maoxin123** also for the invsqrt base and the 4,089-Toffoli
chemistry loaders; **Heliosli** for the chemistry weight update and the public *qcc-circuits-live* dataset;
**prvsiyan** for the chemistry refit and its careful explanations; **manwithacat (James Barlow)** for the HW-25 join and
the *VQCC public circuits* dataset that the landscapes and the timeline below are drawn from; **Lera (marculera)** for
the DF-THC table and words; **Ertuğrul Özer (ozertuu)** for the pinned cache and table; the team
**i want to run gemma 9b** for the `cd71c482` decoder line; **Oleg Sher (sheroleg)**, whose *Quantum Circuits from
Zero* helped us read the invsqrt circuit; and Google Quantum AI and Kaggle for the competition, the open kickmix
simulator and the verifiers. The full list, with every change and link, is in [section 9](#s9). The public circuits are also indexed, one row per
file with its first source and author, in our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) dataset. If we missed you or
misattributed a change, please tell us in the comments and we will fix it.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #eef4fc;"><th style="text-align: left; padding: 5px 8px;">Task</th><th style="text-align: left; padding: 5px 8px;">Portfolio file</th><th style="text-align: left; padding: 5px 8px;">Built by (latest change first)</th></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="padding: 5px 8px;"><code>9c708f7b</code></td><td style="text-align: left; padding: 5px 8px;">our splice (asteriospanoras) of <a href="https://www.kaggle.com/code/wcao24/vqcc-deterministic-open-portfolio">WCao24</a>'s 44-CSWAP normaliser <code>94b29cc5</code> and <a href="https://www.kaggle.com/code/thomasgamet/quantum-attempt-new-base-bundle">Thomas Gamet</a>'s early prefix <code>5ec4f516</code>, on <a href="https://www.kaggle.com/code/maoxin123/lb-25-303-combined-invsqrt-refit-chem">maoxin123</a>'s and <a href="https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook">manwithacat</a>'s 62-qubit line, with <a href="https://www.kaggle.com/code/gengsr/lb-25-2268-shifter-46-own-chemistry-loaders">Gengsr</a>'s shifter</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="padding: 5px 8px;"><code>4c81b726</code></td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/code/maoxin123/qcc-v22-a-392k-toffoli-dqi-decoder">maoxin123</a>'s 392k-Toffoli gate-level rewrite of <code>cd71c482</code> (team <i>i want to run gemma 9b</i> and the decoder line; <a href="https://www.kaggle.com/datasets/maoxin123/qcc-dqi-392k-toffoli-decoder">dataset</a>, CC0). Earlier public bests: <a href="https://www.kaggle.com/code/areebbb/product-retention-133-free-toffolis-lb-25-40767">Areeb Ali</a>'s product retention <code>8eccde65</code>, <a href="https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5">Atakan Aldemir</a>'s GF(64) character <code>eae396ba</code> (on our readout-window transplant onto Atakan's <a href="https://www.kaggle.com/code/atakanaldemir/qcc-shared-products-and-controlled-refits"><code>bb4287e7</code></a>)</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #1baf7a;">chemistry</b></td><td style="padding: 5px 8px;"><code>e9cffb84</code> + 15 loaders</td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/code/maoxin123/qcc-v22-4089-toffoli-chemistry-loaders">maoxin123</a>'s v22 loaders (4,089 Toffolis per pass); <a href="https://www.kaggle.com/datasets/heliosli/qcc-circuits-live">Heliosli</a>'s update of <a href="https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization">prvsiyan</a>'s weights, on <a href="https://www.kaggle.com/datasets/marculera/dfthc-c44-build">Lera (marculera)</a>'s (8, 143, 44) table</td></tr>
<tr><td style="padding: 5px 8px;"><b>this notebook</b></td><td style="padding: 5px 8px;">analysis</td><td style="text-align: left; padding: 5px 8px;"><code>kmsim</code>, the figures, the identities and the measurements (asteriospanoras), building on <a href="https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained">Oleg Sher</a>'s explanations and <a href="https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits">manwithacat's public-circuits dataset</a> and our <a href="https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas">QCC Circuit Atlas</a></td></tr>
</table>

<div style="font-size: 12px; color: #52514e;">The table describes the portfolio at the time of writing (2026-11-12); the code below always reads the
portfolio that is actually attached, and prints its file hashes.</div>

### Setup
The first cell needs kickmix and the competition data: it copies the solution notebook's `submission.zip` and
official-verifier receipt, and exports the circuits we look at to the text format `.kmx`. Every later cell is plain
Python (numpy + matplotlib) on those files.

In [ ]:
# Runs on Kaggle (needs kickmix and the competition data).
import sys, os, pathlib, zipfile, json, time, re, hashlib, shutil, math, difflib, heapq, collections, csv
import numpy as np
import matplotlib.pyplot as plt
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
COMP = pathlib.Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
KM = COMP / "kickmix"; VER = COMP / "verifiers"
sys.path.insert(0, str(KM))
import kickmix as km
inp = pathlib.Path("/kaggle/input"); WK = pathlib.Path("/kaggle/working"); TMP = pathlib.Path("/tmp/a"); TMP.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
# Same submission as the solution notebook: copy its output byte for byte.
cands = sorted((p for p in inp.rglob("submission.zip") if (p.parent / "receipt.json").exists()), key=lambda p: ("best-portfolio" not in str(p), str(p)))
subzip = cands[0]  # the attached solution notebook (its receipt also carries `dqi_salts`, three official DQI runs)
shutil.copy(subzip, WK / "submission.zip")
receipt = json.load(open(subzip.parent / "receipt.json"))
DQI_SALTS = receipt.get("dqi_salts", {})  # optional key: absent in receipts of older solution notebooks
with zipfile.ZipFile(WK / "submission.zip") as z: z.extractall(TMP / "sub")
log("submission.zip copied", sha(WK / "submission.zip")[:12], {k: v[:8] for k, v in receipt["sha256"].items() if "angle" not in k})
def to_kmx(src, dst):
    km.Circuit.from_file(src, format=pathlib.Path(src).suffix[1:]).to_file(dst, format="kmx"); return dst
INV = to_kmx(TMP / "sub" / "invsqrt.kmb", TMP / "invsqrt.kmx")
BASE = to_kmx(COMP / "sample_submission" / "invsqrt.kmb", TMP / "invsqrt_reference.kmx")
CHEM_KMX = [to_kmx(p, TMP / f"{p.stem}.kmx") for p in sorted((TMP / "sub" / "dfthcsos").glob("angle_k_*.kmb"), key=lambda p: int(p.stem.split("_")[-1]))]
PUB_KMX = {}
for p in sorted(inp.rglob("*.kmb")):
    if "/invsqrt/" in str(p) and p.stem[:12] not in PUB_KMX: PUB_KMX[p.stem[:12]] = to_kmx(p, TMP / f"pub_{p.stem[:12]}.kmx")
man = list(csv.DictReader(open(next(p for p in inp.rglob("manifest.csv") if "vqcc-apache" in str(p).lower() or "public-circuits" in str(p).lower()))))  # manwithacat's dataset (the Atlas has its own columns)
log(len(PUB_KMX), "public invsqrt circuits;", len(man), "manifest records;", len(CHEM_KMX), "chemistry loaders")

### Toolbox: `kmsim` (explained in [section 3](#s3))
Semantics from kickmix's `op_type.h` (open source in Qualtran): `HMR` = Hadamard, measure, reset (random outcome m,
phase (-1)^(m v), qubit -> 0, bit <- m); `R` = reset (an error if the qubit was 1); an operation runs only if every
pushed condition bit and its own `if b` bit are 1; `Z_POW` angles are half-turns. The phase is tracked in units of
2^-64 turns, like the verifier. Copy this cell to use it elsewhere (Apache-2.0).

In [ ]:
"""kmsim: pure-Python/numpy kickmix-subset simulator for analysis (not the official verifier).

Semantics from kickmix op_type.h: HMR = Hadamard+measure+reset (outcome random m, phase (-1)^(m*v), qubit -> 0,
bit <- m); R = reset (error if qubit was 1); conditions: op runs only if all pushed condition bits are 1 and its own
`if b` bit is 1. Phase tracked in units of 2^-64 turns (uint64), Z_POW angles are half-turns.
"""
from fractions import Fraction
import numpy as np

MASK = (1 << 64) - 1
TOFF = {"CCX", "CCZ", "CSWAP"}


def half_turns_units(s):
    f = Fraction(s)  # exact decimal
    return int(round(f * (1 << 63))) & MASK  # half-turn = 2^63 units


def parse(path):
    ops = []
    regs = {}
    for line in open(path):
        t = line.split()
        if not t:
            continue
        if t[0] == "APPEND_TO_REGISTER":
            regs.setdefault(t[2], []).append(int(t[1][1:])); continue
        if t[0] == "REGISTER":
            continue
        cond = None
        if "if" in t:
            i = t.index("if"); cond = int(t[i + 1][1:]); t = t[:i]
        name = t[0]; args = t[1:]
        qs = [int(a[1:]) for a in args if a.startswith("q")]
        bs = [int(a[1:]) for a in args if a.startswith("b")]
        ang = None
        if name == "Z_POW":
            ang = half_turns_units(args[1])
        ops.append((name, qs, bs, cond, ang))
    return ops, regs


class Sim:
    def __init__(self, ops, nq, nb, batch, seed=0):
        self.ops = ops; self.B = batch
        self.q = np.zeros((nq, batch), dtype=bool)
        self.b = np.zeros((nb, batch), dtype=bool)
        self.ph = np.zeros(batch, dtype=np.uint64)
        self.rng = np.random.default_rng(seed)
        self.toff = np.zeros(batch, dtype=np.int64)
        self.rot = np.zeros(batch, dtype=np.int64)
        self.bad_reset = np.zeros(batch, dtype=bool)

    def run(self, start=0, stop=None, hook=None):
        stack = getattr(self, "stack", [])
        q, b = self.q, self.b
        ones = np.ones(self.B, dtype=bool)
        for k in range(start, len(self.ops) if stop is None else stop):
            name, qs, bs, cond, ang = self.ops[k]
            if name == "PUSH_CONDITION":
                stack.append(cond); continue
            if name == "POP_CONDITION":
                stack.pop(); continue
            act = ones
            if stack or cond is not None:
                act = ones.copy()
                for c in stack: act &= b[c]
                if cond is not None: act &= b[cond]
            if name == "CX":
                q[qs[1]] ^= q[qs[0]] & act
            elif name == "CZ":
                self._neg(q[qs[0]] & q[qs[1]] & act)
            elif name == "Z":
                self._neg(q[qs[0]] & act)
            elif name == "X":
                q[qs[0]] ^= act
            elif name == "CCX":
                q[qs[2]] ^= q[qs[0]] & q[qs[1]] & act; self.toff += act
            elif name == "CCZ":
                self._neg(q[qs[0]] & q[qs[1]] & q[qs[2]] & act); self.toff += act
            elif name == "SWAP":
                a, c = q[qs[0]].copy(), q[qs[1]].copy()
                q[qs[0]] = np.where(act, c, a); q[qs[1]] = np.where(act, a, c)
            elif name == "CSWAP":
                m = act & q[qs[0]]
                a, c = q[qs[1]].copy(), q[qs[2]].copy()
                q[qs[1]] = np.where(m, c, a); q[qs[2]] = np.where(m, a, c); self.toff += act
            elif name == "NEG":
                self._neg(act)
            elif name == "HMR":
                m = self.rng.random(self.B) < 0.5
                self._neg(m & q[qs[0]] & act)
                q[qs[0]] &= ~act
                b[bs[0]] = np.where(act, m, b[bs[0]])
            elif name == "R":
                self.bad_reset |= q[qs[0]] & act
                q[qs[0]] &= ~act
            elif name == "Z_POW":
                m = q[qs[0]] & act
                self.ph = self.ph + np.where(m, np.uint64(ang), np.uint64(0)); self.rot += act
            elif name == "BIT_STORE0":
                b[bs[0]] &= ~act
            elif name == "BIT_STORE1":
                b[bs[0]] |= act
            elif name == "BIT_INVERT":
                b[bs[0]] ^= act
            elif name in ("INFO_TAG", "DEBUG_PRINT"):
                pass
            else:
                raise ValueError(name)
            if hook is not None:
                hook(k, self)
        self.stack = stack

    def _neg(self, m):
        self.ph = self.ph + np.where(m, np.uint64(1 << 63), np.uint64(0))


def target_and_tol(xs, precision_bits=24):
    import math
    rs = [math.isqrt((1 << 130) // int(x)) for x in xs]
    target = np.array([((r + 1) >> 1) & MASK for r in rs], dtype=np.uint64)
    tol = np.array([min(r >> (precision_bits + 1), 1 << 63) for r in rs], dtype=np.uint64)
    return target, tol


def check(path, xs, seed=0, ops_regs=None):
    ops, regs = ops_regs or parse(path)
    nq = 1 + max(max(o[1]) for o in ops if o[1])
    nb = 1 + max((max(o[2]) for o in ops if o[2]), default=0)
    nb = max(nb, 1 + max((o[3] for o in ops if o[3] is not None), default=0))
    s = Sim(ops, nq, nb, len(xs), seed)
    xq = regs["r0"]
    for i, qi in enumerate(xq):
        s.q[qi] = (xs >> i) & 1 == 1
    s.run()
    xr = sum((s.q[qi].astype(np.uint64) << np.uint64(i)) for i, qi in enumerate(xq))
    restored = np.array_equal(xr, xs.astype(np.uint64))
    scratch = [i for i in range(nq) if i not in xq]
    dirty = s.q[scratch].any(axis=0)
    target, tol = target_and_tol(xs)
    err = (s.ph - target).view(np.int64)
    used = np.abs(err).astype(np.float64)
    return dict(nq=nq, restored=restored, dirty=int(dirty.sum()), bad_reset=int(s.bad_reset.sum()),
                worst_ratio=float((used / tol.astype(np.float64)).max()), toff_mean=float(s.toff.mean()),
                rot_mean=float(s.rot.mean())), s

In [ ]:
# Style and drawing helpers: one palette for the whole notebook (checked for colour-vision deficiency), recessive axes,
# titles that state the takeaway, and a small quantum-circuit drawer for the identities below.
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle
from matplotlib.ticker import FixedLocator, FormatStrFormatter, FuncFormatter
INK, INK2, MUTED, RULE, SURF, NAVY = "#0b0b0b", "#52514e", "#8a8984", "#d6d5d0", "#fcfcfb", "#0b1f4b"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
TC = {"invsqrt": C[0], "dqi": C[1], "dfthcsos": C[2], "chemistry": C[2]}
TN = {"invsqrt": "invsqrt", "dqi": "DQI", "dfthcsos": "chemistry"}
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "axes.edgecolor": "#b9b8b3",
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
                     "grid.color": "#e8e7e3", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
                     "lines.linewidth": 2, "legend.frameon": False, "axes.axisbelow": True, "figure.dpi": 110})

def headline(fig, title, sub=None):
    """Figure title that states the takeaway (left aligned), with an optional grey subtitle."""
    fig.text(0.012, 0.985, title, ha="left", va="top", fontsize=13.5, fontweight="bold", color=INK)
    if sub: fig.text(0.012, 0.985 - 0.30 / fig.get_figheight(), sub, ha="left", va="top", fontsize=9.5, color=INK2)
    top = 1 - (0.62 if sub else 0.40) / fig.get_figheight()
    fig.tight_layout(rect=(0, 0, 1, top))

def draw_circuit(ax, wires, ops, ncols, toff=C[1], title=None):
    """Draw a small circuit. wires: labels top to bottom ('b:...' = classical bit). ops: (col, NAME, *wires[, {'if': bit}])."""
    y = {w: -i for i, w in enumerate(wires)}
    for w in wires:
        if w.startswith("b:"):
            for d in (-0.035, 0.035): ax.plot([-0.4, ncols - 0.4], [y[w] + d] * 2, color=MUTED, lw=0.8, zorder=1)
        else: ax.plot([-0.4, ncols - 0.4], [y[w]] * 2, color=INK2, lw=1.1, zorder=1)
        ax.text(-0.55, y[w], w[2:] if w.startswith("b:") else w, ha="right", va="center", fontsize=10, color=INK2 if w.startswith("b:") else INK)
    dot = lambda x, w, col: ax.add_patch(Circle((x, y[w]), 0.085, color=col, zorder=4))
    def plus(x, w, col):
        ax.add_patch(Circle((x, y[w]), 0.19, facecolor=SURF, edgecolor=col, lw=1.4, zorder=4))
        ax.plot([x - 0.19, x + 0.19], [y[w]] * 2, color=col, lw=1.2, zorder=5); ax.plot([x, x], [y[w] - 0.19, y[w] + 0.19], color=col, lw=1.2, zorder=5)
    def cross(x, w, col):
        for s in (1, -1): ax.plot([x - 0.12, x + 0.12], [y[w] - 0.12 * s, y[w] + 0.12 * s], color=col, lw=1.6, zorder=5)
    def box(x, w, label, col=INK, fc=SURF, wd=0.62):
        ax.add_patch(FancyBboxPatch((x - wd / 2, y[w] - 0.24), wd, 0.48, boxstyle="round,pad=0.02", facecolor=fc, edgecolor=col, lw=1.2, zorder=4))
        ax.text(x, y[w], label, ha="center", va="center", fontsize=7.5, color=col, zorder=5, fontweight="bold")
    n_toff = 0
    for op in ops:
        kw = op[-1] if isinstance(op[-1], dict) else {}
        args = op[2:-1] if kw else op[2:]
        x, name = op[0], op[1]
        col = toff if name in ("CCX", "CCZ", "CSWAP") else INK
        n_toff += name in ("CCX", "CCZ", "CSWAP")
        ys = [y[a] for a in args]
        if len(ys) > 1: ax.plot([x, x], [min(ys), max(ys)], color=col, lw=1.4, zorder=3)
        if name in ("CX", "CCX"): [dot(x, a, col) for a in args[:-1]]; plus(x, args[-1], col)
        elif name in ("CZ", "CCZ"): [dot(x, a, col) for a in args]
        elif name == "CSWAP": dot(x, args[0], col); cross(x, args[1], col); cross(x, args[2], col)
        elif name == "SWAP": cross(x, args[0], col); cross(x, args[1], col)
        elif name == "X": plus(x, args[0], col)
        elif name in ("Z", "NOT", "R"): box(x, args[0], name, wd=0.42)
        elif name == "ZPOW": box(x, args[0], args[1], col=C[6], wd=0.9)
        elif name == "HMR":
            box(x, args[0], "HMR", col=C[0], wd=0.86); yb = y[args[1]]
            for d in (-0.03, 0.03): ax.plot([x + d, x + d], [y[args[0]] - 0.24, yb + 0.08], color=C[0], lw=0.8, zorder=3)
            ax.plot([x], [yb + 0.06], marker="v", color=C[0], ms=6, zorder=5)
        elif name == "BOX":
            lo, hi = min(ys), max(ys)
            ax.add_patch(FancyBboxPatch((x - 0.42, lo - 0.3), 0.84, hi - lo + 0.6, boxstyle="round,pad=0.02", facecolor="#eef4fc", edgecolor=C[0], lw=1.0, zorder=4))
            ax.text(x, (lo + hi) / 2, kw.get("label", ""), ha="center", va="center", fontsize=7.5, color=NAVY, zorder=5)
        if "if" in kw:
            yb, yg = y[kw["if"]], (min(ys) if y[kw["if"]] < min(ys) else max(ys))
            for d in (-0.03, 0.03): ax.plot([x + d, x + d], [yb, yg], color=INK2, lw=0.9, ls=(0, (3, 2)), zorder=2)
            ax.add_patch(Rectangle((x - 0.07, yb - 0.07), 0.14, 0.14, color=INK2, zorder=5))
    ax.set_xlim(-2.0, ncols - 0.2); ax.set_ylim(-len(wires) + 0.35, 0.75); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, loc="left", fontsize=10.5, color=INK, pad=4)
    return n_toff

def run_text(txt, init, nq, nb, batch=64, seed=0):
    """Run a small kmx text circuit in kmsim from basis state `init` (list of bits per qubit) on `batch` shots."""
    ops_ = [ln for ln in txt.strip().splitlines()]
    path = TMP / "snippet.kmx"; path.write_text("\n".join(ops_) + "\n")
    o, _ = parse(path); s = Sim(o, nq, nb, batch, seed)
    for i, bit in enumerate(init): s.q[i] = bool(bit)
    s.run(); return s

In [ ]:
# The portfolio's official numbers (from the solution notebook's receipt) and the hosts' reference circuits.
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}  # reference cost = score 1.0
REF_QT = {"invsqrt": (260, 7988.0), "dqi": (1893, 5725461.0), "dfthcsos": (730, 93237.0)}  # paper Sec. III; SCORING_README
R0 = 1.1912604648115832  # DQI repetition factor at zero failed shots out of 102,400 (verifier formula)
v = receipt["verify"]
rows = {  # task: (qubits, Toffolis per shot, cost; DQI priced at zero failed shots)
    "invsqrt": (v["invsqrt"]["num_qubits"], v["invsqrt"]["toffoli_count"], v["invsqrt"]["score"]),
    "dqi": (v["dqi"]["num_qubits"], v["dqi"]["toffoli_count"], v["dqi"]["score"] / v["dqi"]["repetition_factor"] * R0),
    "dfthcsos": (v["dfthcsos"]["num_qubits"], v["dfthcsos"]["toffoli_count"], v["dfthcsos"]["score"]),
}
def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return math.nan
ratios = {k: REF[k] / c for k, (q, t, c) in rows.items()}
gm = math.prod(ratios.values()) ** (1 / 3)
print("portfolio files:", {k: s[:12] for k, s in receipt["sha256"].items() if not k.startswith("dfthcsos/angle")})
print(f"{'task':9s} {'Q':>5s} {'Toffolis/shot':>14s} {'cost':>20s} {'ref/cost':>9s}")
for k, (q, t, c) in rows.items(): print(f"{k:9s} {q:5d} {t:14,.2f} {c:20,.2f} {ratios[k]:9.4f}")
print(f"geometric mean (DQI at zero failed shots): {gm:.5f}; DQI failed shots in this run: "
      f"{v['dqi']['value_errors'] + v['dqi']['phase_errors']}")
for salt, r_ in DQI_SALTS.items():
    print(f"DQI official run, salt {salt:8s}: {r_.get('status')}, {r_.get('num_qubits')} qubits, {r_.get('toffoli_count', math.nan):,.2f} Toffolis, value errors {r_.get('value_errors')}, phase errors {r_.get('phase_errors')}")

<a id="s1"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">1 &middot; The competition in five minutes</div>

You submit three **kickmix circuits**: circuits built only from reversible classical gates, phase gates and
measurement-based uncomputation. Such a circuit maps each input bit string to one output bit string times a phase, so a
classical computer can check it shot by shot, tracking one bit per qubit plus one phase. That is what makes the
competition *verifiable*: the hosts' verifiers simulate your circuit on many inputs and reject it on a single wrong
shot (DQI tolerates rare failures, at a price). No quantum background is needed; the whole game is to make three
fixed computations cheaper in **qubits** and **Toffoli gates**.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #eef4fc;"><th style="text-align: left; padding: 5px 8px;">Task</th><th style="text-align: left; padding: 5px 8px;">Science</th><th style="text-align: left; padding: 5px 8px;">What the circuit computes</th><th style="text-align: left; padding: 5px 8px;">Cost (lower is better)</th><th style="text-align: left; padding: 5px 8px;">Hosts' reference</th></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="text-align: left; padding: 5px 8px;">fusion: Coulomb phase of every electron pair</td><td style="text-align: left; padding: 5px 8px;">|x&#10217;|0&#10217; &rarr; e<sup>2&pi;i/&radic;x</sup>|x&#10217;|0&#10217; for every x in [1, 3,133,452]</td><td style="text-align: left; padding: 5px 8px;">Q &times; C, C = Toffolis + T/4</td><td style="text-align: left; padding: 5px 8px;">260 qubits, C = 7,988</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="text-align: left; padding: 5px 8px;">optimisation: Decoded Quantum Interferometry</td><td style="text-align: left; padding: 5px 8px;">Reed&ndash;Solomon RS(4095, 4025) syndrome decoder over GF(2<sup>12</sup>): from syndromes S0..S69, apply the sign (&minus;1)<sup>&Sigma; popcount(e<sub>j</sub> &amp; M<sub>j</sub>)</sup> and restore everything</td><td style="text-align: left; padding: 5px 8px;">R(&epsilon;<sub>+</sub>) &times; Q<sup>2</sup> &times; T</td><td style="text-align: left; padding: 5px 8px;">1,893 qubits, 5,725,461 Toffolis</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #1baf7a;">chemistry</b><br><code>dfthcsos</code></td><td style="text-align: left; padding: 5px 8px;">FeMoCo (nitrogen fixation): block encoding of the Hamiltonian</td><td style="text-align: left; padding: 5px 8px;">a family of QROM loaders |a&#10217;|z&#10217; &rarr; |a&#10217;|z &oplus; &delta;<sub>k</sub>(a)&#10217; that stream the Givens angles of a DF-THC factorization (also submitted)</td><td style="text-align: left; padding: 5px 8px;">&lambda;<sub>eff</sub> &times; T &times; Q over the whole block encoding</td><td style="text-align: left; padding: 5px 8px;">730 qubits, T = 93,237, &lambda;<sub>eff</sub> = 49.30</td></tr>
</table>

In [ ]:
# Figure: the three contracts at a glance (inputs, the circuit you write, what it must output, how it is tested).
fig, axs = plt.subplots(3, 1, figsize=(10.5, 7.4))
spec = [
    ("invsqrt", [r"$|x\rangle$  22 qubits", r"$|0\cdots0\rangle$  scratch"], "your circuit\n(phase by $1/\\sqrt{x}$)",
     [r"$e^{2\pi i/\sqrt{x}}\,|x\rangle$", r"$|0\cdots0\rangle$"],
     r"every $x$ in $[1,\ 3{,}133{,}452]$, 64 shots each (200.5M shots); phase error $\leq 2^{-24}/\sqrt{x}$ turns"),
    ("dqi", [r"$|S_0..S_{69}\rangle$  70 x 12 qubits", r"$E_0..E_{4094}$  classical record $M$", r"$|0\cdots0\rangle$  scratch"],
     "your decoder\n(Reed-Solomon over GF($2^{12}$))",
     [r"$(-1)^{\,e\cdot M}\,|S\rangle$  (S restored)", r"$M$  (classical)", r"$|0\cdots0\rangle$"],
     "102,400 random error patterns (up to 35 errors); rare failures allowed, priced by R($\\epsilon_+$); no Z_POW"),
    ("dfthcsos", [r"$|a\rangle$  selection (row address)", r"$|z\rangle$  J word registers", r"$|0\cdots0\rangle$  scratch"],
     "your loaders ANGLE$_k$\n(one QROM per batch)",
     [r"$|a\rangle$", r"$|z \oplus \delta_k(a)\rangle$", r"$|0\cdots0\rangle$"],
     "every address, 64 shots, forward and reverse; $\\leq$128 qubits per loader; the factorization sets $\\lambda_{eff}$"),
]
for ax, (task, ins, mid, outs, test) in zip(axs, spec):
    ax.set_xlim(0, 10); ax.set_ylim(-0.9, 2.2); ax.axis("off")
    col = TC[task]; n = len(ins); ys = np.linspace(1.75, 0.35, n) if n > 2 else [1.45, 0.65]
    ax.add_patch(FancyBboxPatch((3.85, 0.15), 2.3, 1.8, boxstyle="round,pad=0.04", facecolor=col, edgecolor=col, alpha=0.95))
    ax.text(5.0, 1.05, mid, ha="center", va="center", color="white", fontsize=10, fontweight="bold")
    for yy, s in zip(ys, ins):
        ax.text(3.55, yy, s, ha="right", va="center", fontsize=10, color=INK); ax.annotate("", (3.85, yy), (3.6, yy), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1))
    for yy, s in zip(ys, outs):
        ax.text(6.45, yy, s, ha="left", va="center", fontsize=10, color=INK); ax.annotate("", (6.42, yy), (6.15, yy), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1))
    ax.text(0.0, 2.1, TN[task], fontsize=12, fontweight="bold", color=col, va="center")
    ax.text(5.0, -0.45, "verifier: " + test, ha="center", va="center", fontsize=9, color=INK2)
headline(fig, "Three fixed computations: you choose the circuit, the verifier checks every shot",
         "inputs on the left, the contract on the right; scratch qubits must always come back to |0>")
plt.show()

### The score
Each task's cost is divided into the hosts' reference cost, and the three ratios are combined with a geometric mean:

$$\text{Score} = \left(\frac{\text{cost}^{\text{ref}}_{\text{invsqrt}}}{\text{cost}_{\text{invsqrt}}}\cdot\frac{\text{cost}^{\text{ref}}_{\text{dqi}}}{\text{cost}_{\text{dqi}}}\cdot\frac{\text{cost}^{\text{ref}}_{\text{chem}}}{\text{cost}_{\text{chem}}}\right)^{1/3}$$

$$\text{cost}_{\text{invsqrt}} = Q\times\Big(\text{Toffolis}+\tfrac{T}{4}\Big),\qquad \text{cost}_{\text{dqi}} = R(\epsilon_+)\,Q^2\,T,\qquad \text{cost}_{\text{chem}} = \lambda_{\text{eff}}\,T\,Q$$

Two consequences drive everything. **A 1% cut in any task's cost is worth the same** (one third of a percent of the
score), whatever the task. And because each cost is a *product*, a 1% cut in any factor (qubits, Toffolis, the
repetition factor, the chemistry normalisation) is worth the same 1% of that task, except DQI qubits, which count
twice (Q squared). Section 2 turns this into exchange rates.

<div style="background: #fff6e8; border-left: 5px solid #eda100; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How a verifier tests a circuit.</b> Each shot writes an input into the registers, runs the circuit in kickmix and
checks three things: the inputs are restored, every scratch qubit is back to 0, and the tracked phase is right.
Inputs and the random outcomes of <code>HMR</code> measurements are seeded from the SHA-256 of your file plus a
<b>secret salt</b> on the leaderboard (an empty salt locally), so any change to a circuit gets fresh tests and the
leaderboard draws differ from yours. invsqrt runs every input 64 times; DQI draws 102,400 random error patterns and
turns its failures into an upper confidence bound &epsilon;<sub>+</sub> (one-sided Clopper&ndash;Pearson at
&delta; = 2<sup>&minus;128</sup>) and a repetition factor R(&epsilon;<sub>+</sub>) = ln 2 / &minus;ln(1/2 + 2&radic;&epsilon;<sub>+</sub>);
chemistry runs every address 64 times, forward then in reverse, and rejects the submission on any failure or if
the encoded Hamiltonian leaves the accuracy gates (20 mHa, &Lambda; &gt; 49.0, E<sub>gap</sub> &gt; 3.914898 Ha).
</div>

The four panels below show what each test means in numbers.

In [ ]:
# Figure: what the verifiers check, in numbers. The DQI functions follow the formulas in verify_submission_dqi.py.
def cp_upper(k, n=102400, log2_delta=-128.0):
    """One-sided Clopper-Pearson upper bound at confidence 1 - 2^log2_delta (as in the DQI verifier)."""
    ld = log2_delta * math.log(2.0)
    if k == 0: return -math.expm1(ld / n)
    j = np.arange(k + 1, dtype=float); lc = np.r_[0.0, np.cumsum(np.log(n - j[1:] + 1.0) - np.log(j[1:]))]
    def tail(p):
        t = lc + j * math.log(p) + (n - j) * math.log1p(-p); m = t.max(); return m + math.log(np.exp(t - m).sum())
    lo, hi = k / n, 1.0
    for _ in range(80):
        mid = 0.5 * (lo + hi); lo, hi = (mid, hi) if tail(mid) > ld else (lo, mid)
    return hi
def rep_factor(eps): return 1.0 if eps == 0 else math.log(2.0) / -math.log(0.5 + 2.0 * math.sqrt(eps))
def dqi_weights(m=4095, l=35, q=4096, r=2016):
    k = np.arange(l + 1, dtype=float); off = np.sqrt(k[1:] * (m - k[1:] + 1)); d = (q - 2 * r) / math.sqrt(r * (q - r))
    w = np.abs(np.linalg.eigh(np.diag(k * d) + np.diag(off, 1) + np.diag(off, -1))[1][:, -1]); return (w / np.linalg.norm(w)) ** 2
assert abs(rep_factor(cp_upper(0)) - R0) < 1e-12
fig, axs = plt.subplots(2, 2, figsize=(10.5, 6.8))
# (a) invsqrt target and tolerance
xg = np.geomspace(1, 3133452, 400); ax = axs[0, 0]
ax.plot(xg, 1 / np.sqrt(xg), color=TC["invsqrt"], label=r"target phase $1/\sqrt{x}$ (turns)")
ax.plot(xg, 2.0 ** -24 / np.sqrt(xg), color=C[3], label=r"tolerance $2^{-24}/\sqrt{x}$")
ax.fill_between(xg, 1e-12, 2.0 ** -24 / np.sqrt(xg), color=C[3], alpha=0.12)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_ylim(1e-11, 3); ax.set_xlabel("input x"); ax.set_ylabel("turns")
wx = v["invsqrt"]["worst_x"]; ax.scatter([wx], [2.0 ** -24 / math.sqrt(wx)], s=50, color=INK, zorder=4)
ax.annotate(f"portfolio's tightest input x = {wx}:\n{100 * v['invsqrt']['worst_error_over_tolerance']:.2f}% of the tolerance used", (wx, 2.0 ** -24 / math.sqrt(wx)), (60, 1e-6), fontsize=8.5, color=INK2, arrowprops=dict(arrowstyle="-", color=MUTED))
ax.legend(loc="upper right", fontsize=8.5); ax.set_title("invsqrt: ~7 significant digits at all 3.1M inputs", loc="left")
# (b) DQI error-weight distribution
p = dqi_weights(); ax = axs[0, 1]; kk = np.arange(len(p))
ax.bar(kk, p, color=TC["dqi"], width=0.8); ax.set_xlabel("number of errors in a test shot"); ax.set_ylabel("probability")
ax.axvline((p * kk).sum(), color=INK2, lw=1, ls="--"); ax.text((p * kk).sum(), p.max() * 1.02, f" mean {(p * kk).sum():.2f}", color=INK2, fontsize=8.5, ha="right", va="bottom")
ax.set_title("DQI: most shots carry 29-34 errors (max 35)", loc="left"); ax.set_ylim(0, p.max() * 1.15); ax.set_xlim(14.5, 36)
# (c) DQI repetition factor by failed shots
ax = axs[1, 0]; ks = np.arange(6); Rk = np.array([rep_factor(cp_upper(int(k))) for k in ks])
extra = 100 * (Rk / Rk[0] - 1)
ax.bar([str(k) for k in ks], extra, color=TC["dqi"], width=0.6); ax.set_ylim(0, extra.max() * 1.3)
for k, (r, e) in enumerate(zip(Rk, extra)): ax.text(k, e, f"R = {r:.4f}\n{e:+.2f}%" if k else f"R = {r:.4f}\n(zero failures)", ha="center", va="bottom", fontsize=8, color=INK)
ax.set_xlabel("failed shots out of 102,400"); ax.set_ylabel("DQI cost above zero failures (%)")
ax.set_title("DQI: R = 1.19 at best; each failure adds ~0.4%", loc="left")
# (d) chemistry: lambda_eff at the E_gap floor
ax = axs[1, 1]; mt = v["dfthcsos"].get("metrics", {})
Lg = np.linspace(200, 330, 200); EG = 3.914898
ax.plot(Lg, np.sqrt(EG * (2 * Lg - EG)), color=TC["dfthcsos"], label=None); ax.text(236, 40.0, r"curve: $E_{gap}$ at its floor 3.914898 Ha", fontsize=8.5, color=INK2)
pts = [("hosts' baseline", 308.4559397330944, 3.965477244291833, MUTED)]
if "Lambda" in mt: pts.append(("portfolio", mt["Lambda"], mt["E_gap"], INK))
for lab, L_, E_, col in pts:
    le = math.sqrt(E_ * (2 * L_ - E_)); ax.scatter([L_], [le], s=55, color=col, zorder=4)
    ax.annotate(f"{lab}: Lambda {L_:.1f}\nE_gap {E_:.9f}\nlambda_eff {le:.3f}", (L_, le), (285, 42.3) if lab == "portfolio" else (205, 49.6), fontsize=8, color=INK2, va="center", arrowprops=dict(arrowstyle="-", color=MUTED))
ax.set_xlabel(r"one-norm $\Lambda$ of the factorization (Ha)"); ax.set_ylabel(r"$\lambda_{eff}$ (Ha)")
ax.set_title(r"chemistry: a smaller one-norm $\Lambda$ gives a smaller $\lambda_{eff}$", loc="left")
headline(fig, "What the three verifiers ask for, in numbers", "computed from the verifiers' formulas; portfolio values from the official receipt")
plt.show()
if "E_gap" in mt:
    print(f"chemistry gates of the portfolio: error_1 {1000 * mt['error_1']:.6f} mHa, error_2 {1000 * mt['error_2']:.6f} mHa (limit 20), "
          f"E_gap {mt['E_gap']:.9f} Ha (floor 3.914898, strict), Lambda {mt['Lambda']:.3f} (floor 49.0)")
print("""Reading: every chemistry gate is pressed against its limit: the refits push the errors to just under 20 mHa and the
gap to just above its floor, because lambda_eff grows with E_gap. The score then rewards a smaller one-norm Lambda.""")

### The kickmix instruction set, on one card
A circuit is a text file (`.kmx`) or its binary form (`.kmb`), one instruction per line, for example `CCX q4 q5 q51`
or `CZ q0 q1 if b3`. Qubits are `q`, classical bits are `b`. Only three instructions cost anything.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #eef4fc;"><th style="text-align: left; padding: 5px 8px;">Class</th><th style="text-align: left; padding: 5px 8px;">Instruction</th><th style="text-align: left; padding: 5px 8px;">Effect on a basis state</th><th style="text-align: left; padding: 5px 8px;">Price in the score</th></tr>
<tr><td style="text-align: left; padding: 5px 8px;">reversible classical</td><td style="text-align: left; padding: 5px 8px;"><code>X</code>, <code>CX</code>, <code>SWAP</code></td><td style="text-align: left; padding: 5px 8px;">flip / XOR / exchange bits</td><td style="text-align: left; padding: 5px 8px;">free</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;"></td><td style="text-align: left; padding: 5px 8px;"><code>CCX</code> (Toffoli), <code>CSWAP</code> (Fredkin)</td><td style="text-align: left; padding: 5px 8px;">t &oplus;= a&middot;b; swap a, b when c = 1</td><td style="text-align: left; padding: 5px 8px;"><span style="background: #eb6834; color: #fff; border-radius: 8px; padding: 1px 7px;">1 Toffoli</span> each time it runs</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;">diagonal phase</td><td style="text-align: left; padding: 5px 8px;"><code>Z</code>, <code>CZ</code>, <code>NEG</code></td><td style="text-align: left; padding: 5px 8px;">multiply the phase by &minus;1 when all named qubits are 1 (NEG: always)</td><td style="text-align: left; padding: 5px 8px;">free</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;"></td><td style="text-align: left; padding: 5px 8px;"><code>CCZ</code></td><td style="text-align: left; padding: 5px 8px;">&minus;1 when a = b = c = 1</td><td style="text-align: left; padding: 5px 8px;"><span style="background: #eb6834; color: #fff; border-radius: 8px; padding: 1px 7px;">1 Toffoli</span></td></tr>
<tr><td style="text-align: left; padding: 5px 8px;"></td><td style="text-align: left; padding: 5px 8px;"><code>Z_POW q t</code></td><td style="text-align: left; padding: 5px 8px;">phase e<sup>i&pi;t</sup> when q = 1 (t in half-turns)</td><td style="text-align: left; padding: 5px 8px;">invsqrt only: free if t is a multiple of 1/2; 1 T (= &frac14; Toffoli) if a multiple of 1/4; otherwise a <b>rotation</b> of 0.56&nbsp;log<sub>2</sub>(1/&epsilon;)&nbsp;+&nbsp;5.3 T. Forbidden in DQI and chemistry</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;">measurement</td><td style="text-align: left; padding: 5px 8px;"><code>HMR q b</code> (&ldquo;hammer&rdquo;)</td><td style="text-align: left; padding: 5px 8px;">X-basis measurement: random bit m into b, phase (&minus;1)<sup>m&middot;q</sup>, q reset to 0</td><td style="text-align: left; padding: 5px 8px;">free</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;"></td><td style="text-align: left; padding: 5px 8px;"><code>R q</code></td><td style="text-align: left; padding: 5px 8px;">reset a qubit that is already 0 (if it was 1, the shot fails)</td><td style="text-align: left; padding: 5px 8px;">free</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;">classical bits</td><td style="text-align: left; padding: 5px 8px;"><code>BIT_STORE0/1</code>, <code>BIT_INVERT</code></td><td style="text-align: left; padding: 5px 8px;">set or flip a classical bit</td><td style="text-align: left; padding: 5px 8px;">free</td></tr>
<tr><td style="text-align: left; padding: 5px 8px;">control</td><td style="text-align: left; padding: 5px 8px;"><code>... if b</code>, <code>PUSH_CONDITION b</code> / <code>POP_CONDITION</code></td><td style="text-align: left; padding: 5px 8px;">run only when the bit(s) are 1</td><td style="text-align: left; padding: 5px 8px;">a conditioned Toffoli is counted only on the shots where it runs: the cost is a <b>mean per shot</b></td></tr>
</table>

### The one trick everyone uses: measurement-based uncomputation
A scratch qubit `t` that holds `a AND b` must be cleaned. Running the Toffoli again costs a second Toffoli. Instead,
`HMR t` measures it in the X basis: `t` is reset for free, but the state picks up the sign (&minus;1)<sup>m&middot;ab</sup>
for a random bit `m`. A free `CZ a b if m` removes exactly that sign. Half the Toffolis, same result; the verifier sees
a wrong sign on half the shots if the fix-up is missing. The figure draws both versions, and `kmsim` checks them on
every input and both measurement outcomes.

In [ ]:
# Figure + proof: compute-copy-uncompute with 2 Toffolis versus measurement-based uncomputation with 1 Toffoli.
fig, axs = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [4.8, 5.8, 5.8]})
W = ["a", "b", "t", "u", "b:m"]
n1 = draw_circuit(axs[0], W[:4], [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "CCX", "a", "b", "t")], 3)
n2 = draw_circuit(axs[1], W, [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "HMR", "t", "b:m"), (3, "CZ", "a", "b", {"if": "b:m"})], 4)
n3 = draw_circuit(axs[2], W, [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "HMR", "t", "b:m")], 4)
axs[0].set_title(f"uncompute by recomputing: {n1} Toffolis", loc="left", fontsize=10)
axs[1].set_title(f"measurement-based: {n2} Toffoli", loc="left", fontsize=10)
axs[2].set_title("fix-up forgotten: wrong sign", loc="left", fontsize=10, color=C[7])
headline(fig, "Measurement-based uncomputation halves the cost of a temporary AND",
         "all three copy a AND b into u and must leave t = 0; orange = Toffoli; the dashed line is a classical condition on bit m")
plt.show()
txt = {"recompute": "CCX q0 q1 q2\nCX q2 q3\nCCX q0 q1 q2", "measurement-based": "CCX q0 q1 q2\nCX q2 q3\nHMR q2 b0\nCZ q0 q1 if b0",
       "fix-up forgotten": "CCX q0 q1 q2\nCX q2 q3\nHMR q2 b0"}
for name, t_ in txt.items():
    bad = toffs = 0
    for a in (0, 1):
        for b in (0, 1):
            s = run_text(t_, [a, b, 0, 0], 4, 1, batch=256, seed=a * 2 + b)
            ok = (~s.q[2]) & (s.q[3] == bool(a & b)) & (s.ph == 0)
            bad += int((~ok).sum()); toffs = float(s.toff.mean())
    print(f"{name:18s}: Toffolis {toffs:.0f}, failed shots {bad} of 1,024 (4 inputs x 256 random outcomes)")

<a id="s2"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">2 &middot; The score, the cost anatomy and the exchange rates</div>

The hosts' reference circuits are honest textbook constructions. The public frontier is now 11 to 65 times cheaper
per task. Because each cost is a product, the improvement of a task is the product of the improvements of its factors:
on a log scale they simply add up, which is how the first figure draws them.

In [ ]:
# Figure: each task's improvement over the reference, split into its multiplicative factors (log scale).
fac = {"invsqrt": [("qubits Q", REF_QT["invsqrt"][0] / rows["invsqrt"][0]), ("Toffoli-eq. C", REF_QT["invsqrt"][1] / v["invsqrt"]["toffoli_equivalent"])],
       "dqi": [("Q squared", (REF_QT["dqi"][0] / rows["dqi"][0]) ** 2), ("Toffolis T", REF_QT["dqi"][1] / rows["dqi"][1])],
       "dfthcsos": [("qubits Q", REF_QT["dfthcsos"][0] / rows["dfthcsos"][0]), ("Toffolis T", REF_QT["dfthcsos"][1] / rows["dfthcsos"][1]),
                    ("lambda_eff", 49.30136925057438 / v["dfthcsos"]["lambda_eff"])]}
fig, ax = plt.subplots(figsize=(10.5, 3.5))
shades = [1.0, 0.62, 0.35]
for i, task in enumerate(["invsqrt", "dqi", "dfthcsos"]):
    x0 = 1.0
    for j, (lab, f) in enumerate(fac[task]):
        ax.barh(i, x0 * f - x0, left=x0, color=TC[task], alpha=shades[j], height=0.56, edgecolor=SURF, linewidth=2)
        xm = math.sqrt(x0 * x0 * f)
        if f > 1.3: ax.text(xm, i, f"{lab}\nx{f:.2f}", ha="center", va="center", fontsize=8.5, color="white" if j == 0 else INK)
        else: ax.text(xm, i + 0.31, f"{lab} x{f:.3f}", ha="center", va="top", fontsize=8, color=INK2)
        x0 *= f
    ax.text(x0 * 1.06, i, f"{ratios[task]:.2f}x", va="center", fontsize=11, fontweight="bold", color=INK)
ax.set_axisbelow(True); ax.set_xscale("log"); ax.set_yticks(range(3)); ax.set_yticklabels([TN[t] for t in ["invsqrt", "dqi", "dfthcsos"]]); ax.invert_yaxis()
ax.set_xlim(1, 130); ax.xaxis.set_major_locator(FixedLocator([1, 2, 5, 10, 20, 50, 100])); ax.xaxis.set_major_formatter(FormatStrFormatter("%gx"))
ax.xaxis.set_minor_locator(FixedLocator([])); ax.set_xlabel("reference cost / portfolio cost (log scale: the factors add up)"); ax.grid(axis="y", visible=False)
headline(fig, f"The portfolio is {min(ratios.values()):.0f}-{max(ratios.values()):.0f}x cheaper than the reference per task: score {gm:.3f}",
         "score = geometric mean of the three bars; DQI priced at zero failed shots (R equal to the reference's)")
plt.show()

### Cost anatomy: where the cost of the portfolio goes
Four bars, one per cost factor that has parts. invsqrt pays for its rotations in Toffoli-equivalents; DQI's Toffolis
are mostly plain `CCX`; and in chemistry **almost half of the Toffolis and most of the qubits are priced by the
verifier's formula** for the state preparations and Givens rotations, not by the submitted loaders.

In [ ]:
# Figure: cost anatomy from the official receipt.
iv, dq, ch = v["invsqrt"], v["dqi"], v["dfthcsos"]; rs = ch["resources"]
rot_T = iv["toffoli_equivalent"] - iv["toffoli_count"] - iv.get("t_count", 0) / 4
socs = dq.get("shot_operation_counts", {}); nsh = dq["verified_shots"]
bars = [
    ("invsqrt: Toffoli-equivalents per shot", TC["invsqrt"], [("Toffolis", iv["toffoli_count"]), (f"{iv['rotation_count']:.0f} rotations", rot_T), ("T gates", iv.get("t_count", 0) / 4)]),
    ("DQI: Toffolis per shot, by gate", TC["dqi"], [(g, socs[g] / nsh) for g in ("CCX", "CSWAP", "CCZ") if g in socs] or [("Toffolis", dq["toffoli_count"])]),
    ("chemistry: Toffolis of one block encoding", TC["dfthcsos"], [("loaders, 2 passes (submitted)", rs["angles_toffoli"]), ("inner state prep (formula)", rs["inner_state_prep_toffoli"]),
                                                     ("Givens rotations (formula)", rs["givens_rotations_toffoli"]), ("outer prep", rs["outer_state_prep_toffoli"])]),
    ("chemistry: qubits", TC["dfthcsos"], [("shared temporary workspace (formula)", rs["temporary_qubits"]), ("angle words J x brot", rs["angles_persistent_qubits"]),
                                           ("inner prep", rs["inner_state_prep_persistent_qubits"]), ("outer prep", rs["outer_state_prep_persistent_qubits"])]),
]
fig, axs = plt.subplots(4, 1, figsize=(10.5, 5.4))
for ax, (lab, col, parts) in zip(axs, bars):
    tot = sum(p_ for _, p_ in parts); x0 = 0; small = []
    for j, (pl, val) in enumerate(parts):
        ax.barh(0, val, left=x0, color=col, alpha=[1.0, 0.62, 0.38, 0.2][j], height=0.6, edgecolor=SURF, linewidth=2)
        if val / tot > 0.07: ax.text(x0 + val / 2, 0, f"{pl}\n{val:,.0f} ({100 * val / tot:.0f}%)", ha="center", va="center", fontsize=8, color="white" if j == 0 else INK)
        else: small.append(f"{pl} {val:,.2f}" if val < 10 else f"{pl} {val:,.0f}")
        x0 += val
    if small: ax.text(tot, 0.36, "  |  ".join(small), ha="right", va="bottom", fontsize=7.5, color=INK2)
    ax.set_xlim(0, tot); ax.set_ylim(-0.4, 0.75); ax.axis("off"); ax.set_title(f"{lab}: {tot:,.2f}" if tot < 1e4 and tot != int(tot) else f"{lab}: {tot:,.0f}", loc="left", fontsize=10)
headline(fig, f"Where the cost goes: the formula carries {100 * rs['heuristic_toffoli'] / ch['toffoli_count']:.0f}% of chemistry's Toffolis, {100 * (rs['heuristic_persistent_qubits'] + rs['temporary_qubits']) / ch['num_qubits']:.0f}% of its qubits",
         f"official receipt; 'formula' = priced by the verifier's model from (R, B, C) and brot; invsqrt rotations = {100 * rot_T / iv['toffoli_equivalent']:.0f}% of its C")
plt.show()
print(f"invsqrt rotations: {iv['rotation_count']:.0f} at eps = {iv['rotation_eps']:.3e}, "
      f"{0.56 * math.log2(1 / iv['rotation_eps']) + 5.3:.2f} T each = {(0.56 * math.log2(1 / iv['rotation_eps']) + 5.3) / 4:.2f} Toffoli-eq.")
print(f"chemistry: T_total = {rs['angles_toffoli']:,.0f} (loaders) + {rs['heuristic_toffoli']:,} (formula); "
      f"Q_total = {rs['persistent_qubits']} persistent + {rs['temporary_qubits']} shared temporary")

### Exchange rates: what one qubit, one Toffoli or one rotation is worth
The first thing to check before any circuit surgery. Removing one unit of a resource cuts that task's cost by a known
fraction; the geometric mean turns it into score points. The dots are drawn on a log scale; the labels also give the
same change in **Toffolis of that task**, the common currency inside a task.

In [ ]:
# Figure + table: exchange rates at the portfolio's operating point.
inv_C, inv_Q = iv["toffoli_equivalent"], iv["num_qubits"]; dqi_T, dqi_Q = dq["toffoli_count"], dq["num_qubits"]
ch_T, ch_Q, ch_l = ch["toffoli_count"], ch["num_qubits"], ch["lambda_eff"]
rot1 = (0.56 * math.log2(1 / iv["rotation_eps"]) + 5.3) / 4
dq_q = 1 - ((dqi_Q - 1) / dqi_Q) ** 2
xr = [  # (task, resource, relative cost change, same change in Toffolis of that task)
    ("invsqrt", "1 qubit", 1 / inv_Q, inv_C / inv_Q), ("invsqrt", "1 rotation", rot1 / inv_C, rot1), ("invsqrt", "1 Toffoli", 1 / inv_C, 1.0),
    ("dqi", "1 qubit (Q squared)", dq_q, dq_q * dqi_T), ("dqi", "1 failed shot fewer in a draw", 1 - R0 / rep_factor(cp_upper(1)), (1 - R0 / rep_factor(cp_upper(1))) * dqi_T),
    ("dqi", "100 Toffolis", 100 / dqi_T, 100.0), ("dqi", "1 Toffoli", 1 / dqi_T, 1.0),
    ("dfthcsos", "1 qubit", 1 / ch_Q, ch_T / ch_Q), ("dfthcsos", "0.01 Ha of lambda_eff", 0.01 / ch_l, 0.01 / ch_l * ch_T),
    ("dfthcsos", "1 loader Toffoli per pass", 2 / ch_T, 2.0), ("dfthcsos", "1 Toffoli", 1 / ch_T, 1.0),
]
pts_ = [gm * ((1 / (1 - rel)) ** (1 / 3) - 1) for _, _, rel, _ in xr]
fig, ax = plt.subplots(figsize=(10.5, 4.6))
for i, ((t_, res_, rel, tof), p_) in enumerate(zip(xr, pts_)):
    ax.plot([1e-6, p_], [i, i], color=RULE, lw=1.2, zorder=1); ax.scatter([p_], [i], s=70, color=TC[t_], edgecolor=SURF, linewidth=1.5, zorder=3)
    ax.text(p_ * 1.25, i, f"+{p_:.5f}   (= {tof:,.1f} Toffolis)" if tof != 1 else f"+{p_:.6f}", va="center", fontsize=8.5, color=INK)
ax.set_yticks(range(len(xr))); ax.set_yticklabels([f"{TN[t_]}: {r_}" for t_, r_, _, _ in xr]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlim(5e-6, 1); ax.set_xlabel("score points gained by removing one unit (log scale)"); ax.grid(axis="y", visible=False)
headline(fig, f"One qubit is worth ~{inv_C / inv_Q:.0f} Toffolis in invsqrt, ~{dq_q * dqi_T:,.0f} in DQI, ~{ch_T / ch_Q:.0f} in chemistry",
         "score points at the portfolio's operating point; the label converts each unit into Toffolis of the same task")
plt.show()
print(f"{'task':9s} {'resource':26s} {'cost change':>12s} {'score points':>13s} {'= Toffolis':>11s}")
for (t_, res_, rel, tof), p_ in zip(xr, pts_): print(f"{t_:9s} {res_:26s} {-100 * rel:11.5f}% {p_:+13.6f} {tof:11.1f}")
print(f"Reading: 0.001 points on this board is about {300 * 0.001 / gm:.4f}% of one task's cost "
      f"({0.003 / gm * dqi_T:,.0f} DQI Toffolis, {0.003 / gm * inv_C:.2f} invsqrt Toffoli-eq., {0.003 / gm * ch_T:.1f} chemistry Toffolis).")

<a id="s3"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">3 &middot; <code style="color: #ffffff; background: transparent;">kmsim</code>: an exact numpy simulator for these circuits</div>

The toolbox cell at the top defines `kmsim`: `parse(path)` reads a `.kmx` file, `Sim` runs a batch of shots as numpy
boolean arrays (one row per wire, one column per shot) and lets you stop after any operation through a `hook`, and
`check(path, xs)` applies the invsqrt verifier's rules: input restored, scratch clean, no bad `R`, phase within
tolerance, mean Toffolis and rotations per shot. It is ~150 lines because the circuits need only the instructions on
the card in section 1. Use it to test circuit surgery in seconds before the official run.

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Validation.</b> On the portfolio's invsqrt circuit, kmsim gives the same qubit count, the same worst error over
tolerance and the same mean Toffolis as the official verifier (which runs 200.5M shots; kmsim reproduces the numbers
on a 16k-shot sample that contains the worst inputs).
</div>

In [ ]:
rng = np.random.default_rng(0)
xs = np.concatenate([np.arange(1, 4001), rng.integers(1, 3133453, 12000), np.array([161, 221, 644, 3099851, 3133452])]).astype(np.int64)
t = time.time(); r_inv, s_inv = check(INV, xs, seed=1); dt = time.time() - t
print("kmsim   :", {k: r_inv[k] for k in ("nq", "restored", "dirty", "worst_ratio", "toff_mean")}, f"({dt:.1f} s for {len(xs):,} shots)")
print("verifier:", {k: v["invsqrt"].get(k) for k in ("num_qubits", "worst_error_over_tolerance", "toffoli_count")})
if "BASE" in globals():
    r_base, _ = check(BASE, xs[:4000], seed=1)
    print("hosts' reference circuit:", {k: r_base[k] for k in ("nq", "worst_ratio", "toff_mean")})

<a id="s4"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">4 &middot; invsqrt deep dive</div>

**The recipe** (the hosts' paper, Sec. III.A): shift x into a fixed range, get a first guess of 1/&radic;x from a
lookup table plus a cubic polynomial, refine it with one Newton&ndash;Raphson step (the fixed-point cousin of the
*Quake III* fast inverse square root), shift back, and kick the result into the phase with `Z_POW` rotations, one
per bit, before cleaning up. The reference does this with 260 qubits. The public 62-qubit line (built by many
authors, section 7) keeps the shape but squeezes every stage; Oleg Sher's *Quantum Circuits from Zero* walks
through it step by step. Below, `kmsim` opens the portfolio's circuit.

In [ ]:
# Figure: the invsqrt pipeline (the hosts' reference recipe), with the reference's size.
fig, ax = plt.subplots(figsize=(10.5, 2.2)); ax.set_xlim(0, 10.5); ax.set_ylim(0, 2); ax.axis("off")
steps = [("|x>", "22-qubit input"), ("normalise", "shift x by 2s\ninto [1, 4)"), ("first guess", "lookup table +\ncubic polynomial"),
         ("Newton step", "y1 = y0 (3 - x y0^2) / 2"), ("shift back", "y = y1 * 2^s"), ("phase kickback", "Z_POW per bit:\ne^(2 pi i y)"), ("uncompute", "scratch -> |0>\n(HMR + fix-ups)")]
for i, (a, b) in enumerate(steps):
    x0 = 0.1 + i * 1.48; col = TC["invsqrt"] if 0 < i < 6 else NAVY
    ax.add_patch(FancyBboxPatch((x0, 0.55), 1.22, 0.95, boxstyle="round,pad=0.03", facecolor=col, edgecolor=col, alpha=0.95 if i in (0, 6) else 0.85))
    ax.text(x0 + 0.61, 1.22, a, ha="center", va="center", color="white", fontsize=9.5, fontweight="bold")
    ax.text(x0 + 0.61, 0.83, b, ha="center", va="center", color="white", fontsize=7.3)
    if i < len(steps) - 1: ax.annotate("", (x0 + 1.46, 1.02), (x0 + 1.24, 1.02), arrowprops=dict(arrowstyle="-|>", color=INK2))
ax.text(0.1, 0.18, "hosts' reference: 260 qubits, 7,735 Toffolis, 37 rotations, C = 7,988.   Portfolio: "
        f"{v['invsqrt']['num_qubits']} qubits, {v['invsqrt']['toffoli_count']:,.1f} Toffolis, {v['invsqrt']['rotation_count']:.0f} rotations, C = {v['invsqrt']['toffoli_equivalent']:,.1f}.", fontsize=8.5, color=INK2)
headline(fig, "invsqrt: a fixed-point fast inverse square root, run reversibly", "the reference recipe (paper Sec. III.A); the public circuits keep this shape")
plt.show()

### One shot, every wire
A kickmix shot is just bits moving on wires. The raster shows all 62 wires (rows) along the circuit (columns) for one
input, the tightest one for the error budget: dark = the wire holds a 1. The 22 input wires stay put at the top; the
scratch wires fill, compute and empty again.

In [ ]:
# Figure: the state of every wire along one shot of the portfolio's invsqrt circuit.
ops, regs = parse(INV)
nb_ = 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0))
nq_ = 1 + max(max(o[1]) for o in ops if o[1])
x_one = v["invsqrt"]["worst_x"]; s1 = Sim(ops, nq_, nb_, 1, 0)
for i, qi in enumerate(regs["r0"]): s1.q[qi] = (x_one >> i) & 1 == 1
step = 8; snaps = []
s1.run(hook=lambda k, sim: snaps.append(sim.q[:, 0].copy()) if k % step == 0 else None)
order = list(regs["r0"]) + [q for q in range(nq_) if q not in regs["r0"]]
img = np.array(snaps).T[order]
fig, ax = plt.subplots(figsize=(10.5, 3.6))
from matplotlib.colors import ListedColormap
ax.imshow(img, aspect="auto", interpolation="nearest", cmap=ListedColormap([SURF, TC["invsqrt"]]), extent=(0, len(ops), nq_ - 0.5, -0.5))
ax.axhline(len(regs["r0"]) - 0.5, color=C[1], lw=1)
ax.set_xlabel("operation index"); ax.set_ylabel("wire"); ax.grid(False)
headline(fig, f"One shot of the invsqrt circuit (x = {x_one}): the scratch wires fill and empty again",
         f"{nq_} wires x {len(ops):,} operations (every {step}th shown); blue = the wire holds 1; above the orange line: the 22 input wires")
plt.show()
print("x restored and scratch clean at the end:", int(sum(int(s1.q[q, 0]) << i for i, q in enumerate(regs["r0"]))) == x_one and not s1.q[order[22:], 0].any())

### Where the Toffoli-equivalents go
The verifier's cost is Q x C with C = mean Toffolis + T/4, every non-Clifford rotation priced at 0.56 log2(1/eps) + 5.3
T, where eps splits the error margin left over by the simulated error evenly across the rotations. Below: cumulative
Toffolis along the circuit, with the rotations marked, and the qubits in use (non-zero on at least one shot).

In [ ]:
xs_a = np.concatenate([np.arange(1, 1501), rng.integers(1, 3133453, 2500)]).astype(np.int64)
s = Sim(ops, nq_, nb_, len(xs_a), 3)
for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs_a >> i) & 1 == 1
cum_t = np.zeros(len(ops)); live = np.zeros(len(ops), dtype=int)
def hook(k, sim):
    cum_t[k] = sim.toff.mean()
    if k % 5 == 0: live[k] = int(sim.q.any(axis=1).sum())
s.run(hook=hook)
cum_t = np.maximum.accumulate(cum_t)  # condition push/pop operations skip the hook
live = np.where(live > 0, live, np.nan).astype(float)
rot_idx = [k for k, o in enumerate(ops) if o[0] == "Z_POW" and o[4] % (1 << 61) != 0]
fig, axs = plt.subplots(2, 1, figsize=(10.5, 4.8), sharex=True)
axs[0].plot(cum_t, color=TC["invsqrt"]); axs[0].set_ylabel("Toffolis per shot")
for k in rot_idx: axs[0].axvline(k, color=C[3], lw=0.7, alpha=0.8)
axs[0].text(rot_idx[0], cum_t[-1] * 0.86, f" {len(rot_idx)} rotations (yellow lines)", color=INK2, ha="right", fontsize=9)
axs[0].set_title("cumulative mean Toffolis", loc="left", fontsize=10)
ok = ~np.isnan(live); axs[1].plot(np.flatnonzero(ok), live[ok], color=C[2], lw=1.2); axs[1].set_ylim(20, nq_ + 2)
axs[1].set_ylabel("wires"); axs[1].set_xlabel("operation index"); axs[1].set_title(f"wires holding data on at least one shot (out of {nq_})", loc="left", fontsize=10)
headline(fig, "invsqrt: the Toffolis are spread along the circuit, the rotations come in two bursts",
         "portfolio circuit, 4,000 shots in kmsim")
plt.show()
n_rot = len(rot_idx); rot_cost = n_rot * (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
print(f"{n_rot} rotations at eps={v['invsqrt']['rotation_eps']:.2e}: {rot_cost:.0f} Toffoli-eq "
      f"= {100 * rot_cost / v['invsqrt']['toffoli_equivalent']:.1f}% of C; one qubit = {100 / nq_:.2f}% of the cost")

### The two phase streams
Almost every rotation acts on one helper wire: a multiplexer (the shifter, a few Toffolis) routes one bit of the value
into it, the rotation fires, and the helper is measured away. The first burst phases a table value, the second the
interpolation product (the two-stream reading follows Oleg Sher's notebook). Their weights overlap on several
positions, which looks like waste: could one stream absorb the other?

In [ ]:
helper = collections.Counter(ops[k][1][0] for k in rot_idx).most_common(1)[0][0]
info = []
for k in rot_idx:
    j = k - 1
    while j > 0 and not (ops[j][0] in ("HMR", "Z_POW") and ops[j][1] == [helper]): j -= 1
    seg = ops[j + 1:k]
    src = next((o[1][0] for o in seg if o[0] == "CX" and o[1][1] == helper), None)
    info.append((k, ops[k][4] / 2**63, sum(o[0] in TOFF for o in seg), src))
w = np.log2([i[1] for i in info])
drops = [i for i in range(1, len(w)) if w[i] < w[i - 1] - 0.5]; split = drops[0] if drops else len(w)
stage = (np.arange(len(w)) >= split).astype(int)
fig, ax = plt.subplots(figsize=(10.5, 2.9))
for st, col, lab in ((0, TC["invsqrt"], "stream 1 (table value)"), (1, C[1], "stream 2 (interpolation product)")):
    m = stage == st; ax.scatter(w[m], [st] * m.sum(), s=60, color=col, label=lab, edgecolor=SURF, linewidth=1.5, zorder=3)
ov = sorted(set(np.round(w[stage == 0])) & set(np.round(w[stage == 1])))
if ov: ax.axvspan(min(ov) - 0.5, max(ov) + 0.5, color=C[3], alpha=0.15, zorder=0); ax.text((min(ov) + max(ov)) / 2, 1.45, f"{len(ov)} weights paid twice", ha="center", fontsize=9, color=INK2)
ax.set_yticks([0, 1]); ax.set_yticklabels(["stream 1\n(table value)", "stream 2\n(interpolation product)"]); ax.set_xlabel("rotation angle, log2(half-turns)")
ax.set_ylim(-0.6, 1.7); ax.grid(axis="y", visible=False)
headline(fig, f"Two phase streams: {len(ov)} rotation weights are paid twice", f"helper wire q{helper}; {len(w)} rotations of the portfolio circuit")
plt.show()
print("overlapping weights (log2 half-turns):", [int(x) for x in ov])
# Is the stream-1 source of each overlapping bit still intact when stream 2 runs?
s2 = Sim(ops, nq_, nb_, 512, 5)
xs_b = rng.integers(1, 3133453, 512).astype(np.int64)
for i, qi in enumerate(regs["r0"]): s2.q[qi] = (xs_b >> i) & 1 == 1
first_stage2 = info[split][0] if split < len(info) else len(ops)
rot_src = {k: src for (k, wt, nt, src), st in zip(info, stage) if st == 0 and src is not None and round(math.log2(wt)) in ov}
snap = {}
def hook2(k, sim):
    if k in rot_src: snap[k] = sim.q[rot_src[k]].copy()
s2.run(0, first_stage2, hook=hook2)
alive = {f"q{rot_src[k]}": bool((val == s2.q[rot_src[k]]).all()) for k, val in snap.items()}
print("stream-1 source wire still holds its bit when stream 2 starts:", alive)
print("""Reading: the low table bits are overwritten before the product exists, so adding them into the product needs those
bits kept alive (more qubits, each worth ~1.6% of the cost) or recomputed. The merge would save a handful of rotations
(~7 Toffoli-eq. each, see the exchange rates), so on this design it does not pay. A design that phases once from one
accumulator would have to be built around it from the start.""")

### Error budget across x
The worst input uses almost all of its tolerance, so the rotations get almost no synthesis budget (their eps is tiny,
which is why each costs ~29 T). Every error source has been squeezed to just under tol(x) = 2^-24 / sqrt(x) turns.

In [ ]:
xs_e = np.unique(np.concatenate([np.arange(1, 3000), np.geomspace(1, 3133452, 6000).astype(np.int64), [v["invsqrt"]["worst_x"]]])).astype(np.int64)
_, se = check(INV, xs_e, seed=2)
target, tol = target_and_tol(xs_e)
ratio = np.abs((se.ph - target).view(np.int64)).astype(float) / tol.astype(float)
fig, ax = plt.subplots(figsize=(10.5, 3.0))
ax.scatter(xs_e, ratio, s=4, color=TC["invsqrt"], alpha=0.5, linewidths=0); ax.set_xscale("log"); ax.axhline(1, color=C[7], lw=1)
ax.text(1.2, 1.02, "fail", color=C[7], fontsize=9, va="bottom")
iw = int(np.argmax(ratio)); ax.scatter([xs_e[iw]], [ratio[iw]], s=50, color=INK, zorder=4); ax.annotate(f"x = {xs_e[iw]}: {ratio[iw]:.4f}", (xs_e[iw], ratio[iw]), (xs_e[iw] * 0.08, 0.86), fontsize=9, color=INK, bbox=dict(fc=SURF, ec=RULE, boxstyle="round,pad=0.25"), arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_xlabel("x"); ax.set_ylabel("|phase error| / tolerance"); ax.set_ylim(0, 1.12)
headline(fig, "invsqrt error budget: the worst input sits at 99.9% of its tolerance", "simulated phase error before rotation synthesis, 9,000 inputs")
plt.show()

### Splicing public circuits at clean cuts (and a sampling trap)
HW-25 (manwithacat) joined two public circuits where they agree on every wire. We generalised it: a cut is *clean*
when the condition stack is empty and no classical bit written before it is read after it; at a clean cut the rest of
the circuit depends only on the wires, and the phase so far is a function of x alone. Two circuits whose clean cuts
hold the same wires and phase on every sampled input can be joined there. A shortest path over all such sync points
of all public 62-qubit invsqrt circuits gives the cheapest splice.

**History.** On the public set of early November, this found nothing beyond HW-25. Once WCao24's normaliser
(`94b29cc5`) and Thomas Gamet's early prefix (`5ec4f516`) were public, the same search found the portfolio's
`9c708f7b`: 1,213.87 Toffolis per shot against 1,216.83 for the best single circuit (official verifier, -0.20% cost).
The cell below runs the search on whatever public set is attached, on a small and a large input sample. **Always
re-measure on a large fresh sample**: conditional blocks fire on random measurement outcomes, so on a small sample
the search picks segments that were cheap by chance.

In [ ]:
# Slow: a few minutes on Kaggle. clean cuts of every public 62-qubit circuit, then a shortest path.
def bits_last_read(ops):
    last = {}
    for k, (n, qs, bs, c, a) in enumerate(ops):
        if c is not None: last[c] = k
        if n == "BIT_INVERT": last[bs[0]] = k
    return last
def clean_cuts(path, xs, seed=7):
    ops, regs = parse(path)
    nb = 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0))
    last = bits_last_read(ops); s = Sim(ops, 62, nb, len(xs), seed)
    for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs >> i) & 1 == 1
    H = lambda: hashlib.blake2b(np.packbits(s.q).tobytes() + s.ph.tobytes(), digest_size=16).hexdigest()
    cuts, open_until, depth, rot = [(0, H(), 0.0, 0)], -1, 0, 0
    for k, (n, qs, bs, c, a) in enumerate(ops):
        depth += (n == "PUSH_CONDITION") - (n == "POP_CONDITION")
        s.run(k, k + 1)
        for b in bs: open_until = max(open_until, last.get(b, -1))
        if n == "Z_POW" and a % (1 << 61): rot += 1
        if depth == 0 and open_until <= k: cuts.append((k + 1, H(), float(s.toff.mean()), rot))
    return cuts
def splice(data, rot_eq):
    by_hash = collections.defaultdict(list)
    for name, cuts in data.items():
        for i, c in enumerate(cuts): by_hash[c[1]].append((name, i))
    start = next(iter(data.values()))[0][1]; dist, pq, best = {}, [], None
    for node in by_hash[start]: dist[node] = 0.0; heapq.heappush(pq, (0.0, *node))
    while pq:
        d, name, i = heapq.heappop(pq)
        if d > dist.get((name, i), 1e18): continue
        cuts = data[name]
        if i == len(cuts) - 1:
            best = min(best or (1e18,), (d, name)); continue
        w_ = cuts[i + 1][2] - cuts[i][2] + rot_eq * (cuts[i + 1][3] - cuts[i][3])
        for nxt, nd in [((name, i + 1), d + w_)] + [(o, d) for o in by_hash[cuts[i][1]] if o != (name, i)]:
            if nd < dist.get(nxt, 1e18): dist[nxt] = nd; heapq.heappush(pq, (nd, *nxt))
    return best
cands = {k: p for k, p in PUB_KMX.items() if 1 + max(max(o[1]) for o in parse(p)[0] if o[1]) == 62}
log(len(PUB_KMX), "public invsqrt circuits,", len(cands), "with 62 qubits")
rot_eq = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
for label, n_rand in (("small sample (512 shots)", 384), ("large sample (4,098 shots)", 3072)):
    xs_s = np.concatenate([np.arange(1, 1 + (128 if n_rand == 384 else 1024)), np.random.default_rng(11).integers(1, 3133453, n_rand), [161, 3133452]]).astype(np.int64)
    data = {k: clean_cuts(p, xs_s) for k, p in cands.items()}
    singles = {k: c[-1][2] + rot_eq * c[-1][3] for k, c in data.items()}
    best = splice(data, rot_eq); b1 = min(singles, key=singles.get)
    print(f"{label}: best single {b1} {singles[b1]:.3f}, best splice {best[0]:.3f} ({best[0] - singles[b1]:+.3f})")
log("splice done")

### The invsqrt landscape: every public circuit in qubits x Toffoli-equivalents
Each point is a public circuit measured with kmsim on the same 4,000 inputs; the grey curves are constant cost
(Q x C). The frontier moved from 260 qubits down to a dense cluster at 62, and since then mostly along the C axis.

In [ ]:
land = []
xs_l = np.concatenate([np.arange(1, 1001), np.random.default_rng(4).integers(1, 3133453, 3000), [161, 3133452]]).astype(np.int64)
rot1 = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
for name, path in list(PUB_KMX.items()) + [("portfolio", INV)]:
    try: r_, s_ = check(path, xs_l, seed=1)
    except Exception: continue
    if not (r_["restored"] and r_["dirty"] == 0 and r_["worst_ratio"] <= 1): continue
    ops_l, _ = parse(path); n_rot_ = sum(1 for o in ops_l if o[0] == "Z_POW" and o[4] % (1 << 61))
    land.append((name, r_["nq"], r_["toff_mean"] + n_rot_ * rot1))
L = np.array([(q, c) for _, q, c in land], dtype=float); is_pf = np.array([n == "portfolio" for n, _, _ in land])
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.9), gridspec_kw={"width_ratios": [1.15, 1]})
for ax, zoom in zip(axs, (False, True)):
    sel = (L[:, 0] == 62) if zoom else np.ones(len(L), bool)
    qq = np.linspace(55, 280, 300) if not zoom else np.array([61.6, 62.4])
    levels = (1e5, 2e5, 4e5, 1e6, 2e6) if not zoom else ()
    y_lo, y_hi = L[:, 1].min() * 0.85, L[:, 1].max() * 1.5
    for sc in levels:
        ax.plot(qq, sc / qq, color=RULE, lw=1)
        if y_lo < sc / 270 < y_hi: ax.text(270, sc / 270, f"Q x C = {sc:,.0f}", color=MUTED, fontsize=7.5, ha="right", va="bottom")
    if zoom:
        C62 = np.sort(L[sel & ~is_pf, 1])[:25]; ax.scatter(np.arange(len(C62)), C62, s=26, color=TC["invsqrt"], edgecolor=SURF, linewidth=1, zorder=3, label="public 62-qubit circuits (sorted)")
        if is_pf.any(): ax.axhline(L[is_pf, 1][0], color=C[1], lw=1.5, label=f"portfolio: C = {L[is_pf, 1][0]:,.1f}")
        ax.set_xlabel("public 62-qubit circuits, cheapest first"); ax.set_ylabel("Toffoli-eq. C"); ax.set_ylim(min(C62.min(), L[is_pf, 1].min() if is_pf.any() else C62.min()) - 4, C62.max() + 3)
        ax.legend(loc="upper left", fontsize=8.5); ax.set_title("the 25 cheapest 62-qubit circuits", loc="left", fontsize=10)
    else:
        ax.scatter(L[~is_pf, 0], L[~is_pf, 1], s=26, color=TC["invsqrt"], edgecolor=SURF, linewidth=1.2, zorder=3, label="public circuit")
        if is_pf.any(): ax.scatter(L[is_pf, 0], L[is_pf, 1], s=80, color=C[1], zorder=4, label="portfolio")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffoli-equivalents C")
        ax.xaxis.set_major_locator(FixedLocator([62, 100, 150, 260])); ax.xaxis.set_major_formatter(FormatStrFormatter("%g")); ax.xaxis.set_minor_locator(FixedLocator([]))
        ax.set_ylim(y_lo, y_hi); ax.set_xlim(55, 280); ax.yaxis.set_major_locator(FixedLocator([1500, 2000, 3000, 5000, 8000])); ax.yaxis.set_major_formatter(FormatStrFormatter("%g")); ax.yaxis.set_minor_locator(FixedLocator([])); ax.legend(loc="upper right", fontsize=8.5); ax.set_title("all valid public circuits; grey = equal Q x C", loc="left", fontsize=10)
headline(fig, "invsqrt landscape: everyone converged on 62 qubits, then fought over C", f"{len(land) - is_pf.sum()} valid public circuits measured with kmsim on 4,000 inputs (manwithacat's public-circuits dataset)")
plt.show()
best_i = int(np.argmin(L[:, 0] * L[:, 1])); print(f"cheapest Q x C (kmsim estimate) = {L[best_i, 0] * L[best_i, 1]:,.0f} ({land[best_i][0]})")

<a id="s5"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">5 &middot; DQI deep dive</div>

**The decoder** (paper Sec. III.B): from the 70 syndromes of a Reed&ndash;Solomon code over GF(2<sup>12</sup>), run the
extended Euclidean algorithm reversibly to get the error locator, sweep the 4,095 positions to recover each error
value and apply its sign, then run everything backwards to restore the syndromes. The reference uses 1,893 qubits and
5.7M Toffolis (63-Toffoli Karatsuba multipliers, Itoh&ndash;Tsujii inversions). The public line is now at 892
qubits and 392k Toffolis per shot (the portfolio file, maoxin123's `4c81b726`), with hundreds of small identities stacked by many authors (section 7). Because the
cost has **Q squared**, one DQI qubit is worth ~880 of its Toffolis.

### 5.1 The readout-window identity (our version-1 change, proof by exhaustion)
A *readout window* is `CSWAP c a b` followed by X-measuring both outputs. The CSWAP costs a Toffoli. The same phase
and the same classical record come from `CX a b; HMR a -> m1; CZ c b if m1; HMR b -> m2; BIT_INVERT m1 if m2`, with
no Toffoli: the phase is m1 a + m1 c (a xor b) + m2 (a xor b) = (m1 xor m2) a' + m2 b', with a' = c ? b : a and
b' = c ? a : b. Atakan Aldemir's `e9f54be3` line had four such windows; we moved them onto `bb4287e7` (-48 Toffolis
per shot). This is the history of an earlier portfolio: the readout windows entered our version-1 DQI file, and the
current portfolio DQI (`4c81b726`, a rewrite of `cd71c482`) does not build on `bb4287e7` or on that transplant.

In [ ]:
# Figure + proof: the readout window before and after; kmsim checks the phase/record relation on all inputs.
fig, axs = plt.subplots(1, 2, figsize=(11, 3.3), gridspec_kw={"width_ratios": [4.6, 7.2]})
n_b = draw_circuit(axs[0], ["c", "a", "b", "b:m1", "b:m2"], [(0, "CSWAP", "c", "a", "b"), (1, "HMR", "a", "b:m1"), (2, "HMR", "b", "b:m2")], 3)
n_a = draw_circuit(axs[1], ["c", "a", "b", "b:m1", "b:m2"], [(0, "CX", "a", "b"), (1, "HMR", "a", "b:m1"), (2, "CZ", "c", "b", {"if": "b:m1"}),
                                                            (3, "HMR", "b", "b:m2"), (4, "NOT", "b:m1", {"if": "b:m2"})], 5)
axs[0].set_title(f"before: {n_b} Toffoli", loc="left", fontsize=10); axs[1].set_title(f"after: {n_a} Toffolis (NOT = BIT_INVERT m1 if m2)", loc="left", fontsize=10)
headline(fig, "A readout window without its Toffoli: swap, then measure = measure, then fix the sign",
         "both leave c unchanged, a and b at 0, and the phase (-1)^(m1 a' + m2 b') of the swapped values")
plt.show()
before = "CSWAP q0 q1 q2\nHMR q1 b0\nHMR q2 b1"
after = "CX q1 q2\nHMR q1 b0\nCZ q0 q2 if b0\nHMR q2 b1\nBIT_INVERT b0 if b1"
ok = True
for c in (0, 1):
    for a in (0, 1):
        for b in (0, 1):
            ap, bp = (b, a) if c else (a, b)
            for txt_ in (before, after):
                s_ = run_text(txt_, [c, a, b], 3, 2, batch=256, seed=4 * c + 2 * a + b)
                expect = ((s_.b[0] & bool(ap)) ^ (s_.b[1] & bool(bp))).astype(np.uint64) << np.uint64(63)
                ok &= bool((s_.ph == expect).all()) and not s_.q[1:].any() and bool((s_.q[0] == bool(c)).all())
print("readout-window identity holds on all 8 inputs x 256 random measurement records:", ok)

### 5.2 Region profile: what each DQI line pays, by decoder subroutine
The DQI files carry `INFO_TAG` markers around subroutines. Counting static Toffolis between markers compares lineages
region by region. This is how we found the two regions where the older `e9f54be3`/`9fcda0c1` line was cheaper than
the `cd71c482` line that `bb4287e7` builds on. The column *portfolio* is the current portfolio file `4c81b726`; its
tags need not match those of the older files one for one, because it is a gate-level rewrite of `cd71c482`.

In [ ]:
# Runs on Kaggle (needs kickmix to export the DQI files, ~60M operations each).
def region_profile(kmx_path, live_trace=False):
    regions, cur, depth = [], {"tag": "<start>", "toff": 0, "ops": 0}, 0
    WRITE = {"X": (1,), "CX": (2,), "CCX": (3,), "SWAP": (1, 2), "CSWAP": (2, 3)}
    zero, cur_live, peak, trace, n = None, 0, 0, [], 0
    for line in open(kmx_path):
        if line.startswith("APPEND_TO_REGISTER"):
            if live_trace and line.split()[1].startswith("q"):
                q = int(line.split()[1][1:]); zero = zero or [True] * 4096
                if zero[q]: zero[q] = False; cur_live += 1
            continue
        if line.startswith("REGISTER"): continue
        if line.startswith("INFO_TAG"):
            regions.append(cur); cur = {"tag": line[9:].strip().strip('"'), "toff": 0, "ops": 0}; continue
        t = line.split(); op = t[0]; n += 1
        depth += (op == "PUSH_CONDITION") - (op == "POP_CONDITION")
        cur["ops"] += 1
        if op in TOFF: cur["toff"] += 1
        if live_trace:
            zero = zero or [True] * 4096
            unc = depth == 0 and "if" not in t
            if op in ("HMR", "R") and unc:
                q = int(t[1][1:])
                if not zero[q]: zero[q] = True; cur_live -= 1
            elif op == "SWAP" and unc:
                a_, b_ = int(t[1][1:]), int(t[2][1:]); zero[a_], zero[b_] = zero[b_], zero[a_]
            elif op in WRITE:
                for i in WRITE[op]:
                    q = int(t[i][1:])
                    if zero[q]: zero[q] = False; cur_live += 1
            if n % 20000 == 0: trace.append((n, cur_live))
            peak = max(peak, cur_live)
    regions.append(cur)
    return regions, trace, peak
dqi_src = {"portfolio": TMP / "sub" / "dqi.kmb"}
for p in inp.rglob("*.kmb"):
    for w_ in ("cd71c4822aa0", "9fcda0c1aeb3"):
        if p.name.startswith(w_): dqi_src[w_[:8]] = p
for z in inp.rglob("submission.zip"):
    if "shared-products" in str(z).lower():
        with zipfile.ZipFile(z) as zz: zz.extract("dqi.kmb", TMP / "bb42")
        dqi_src["bb4287e7"] = TMP / "bb42" / "dqi.kmb"
prof = {}
for k, p in dqi_src.items():
    kx_ = to_kmx(p, TMP / "dqi_tmp.kmx")
    prof[k] = region_profile(kx_, live_trace=(k == "portfolio"))
    log(k, "regions", len(prof[k][0]), "static Toffolis", sum(r["toff"] for r in prof[k][0]))
by_tag = {k: collections.Counter() for k in prof}
for k, (regs_, _, _) in prof.items():
    for r in regs_: by_tag[k][r["tag"].replace("@exit:", "").replace("@enter:", "")] += r["toff"]
top = [t for t, _ in by_tag["portfolio"].most_common(10)]
print("Static Toffolis count every gate in the file, including branches that run only for some measurement outcomes;")
print("the newer files carry larger static blocks but execute fewer Toffolis per shot.")
print(f"{'subroutine (static Toffolis)':32s}" + "".join(f"{k:>12s}" for k in prof))
for t in top: print(f"{t[:32]:32s}" + "".join(f"{by_tag[k][t]:12,d}" for k in prof))
fig, ax = plt.subplots(figsize=(10.5, 3.8))
keys = list(prof); wbar = 0.8 / len(keys)
for j, k in enumerate(keys):
    ax.barh(np.arange(len(top)) + j * wbar, [by_tag[k][t] for t in top], height=wbar, color=C[j % len(C)], label=k, edgecolor=SURF, linewidth=1)
ax.set_yticks(np.arange(len(top)) + 0.4 - wbar / 2); ax.set_yticklabels([t[:30] for t in top]); ax.invert_yaxis(); ax.legend(loc="lower right", fontsize=8.5)
ax.set_xlabel("static Toffolis in the region"); ax.grid(axis="y", visible=False)
headline(fig, "DQI region profile: the same subroutines, different prices per lineage", "static Toffolis between INFO_TAG markers, the 10 largest regions of the portfolio file (4c81b726)")
plt.show()

### 5.3 Qubits in use along the decoder (static liveness)
A wire is counted from its first write until an unconditional `HMR`/`R` clears it. The decoder hits its full width in
hundreds of short episodes inside the GF(2<sup>12</sup>) multipliers, so removing one qubit (worth 0.22% of the DQI
cost, because the cost has Q squared) needs a multiplier that is narrower everywhere, not a local patch.

In [ ]:
# Runs on Kaggle (uses the region profile above).
_, trace, peak = prof["portfolio"]
tr = np.array(trace)
fig, ax = plt.subplots(figsize=(10.5, 3.0))
ax.plot(tr[:, 0] / 1e6, tr[:, 1], color=TC["dqi"], lw=1)
ax.axhline(840, color=INK2, lw=1); ax.text(0, 842, " 840 = syndrome input (cannot shrink)", color=INK2, va="bottom", fontsize=9)
ax.set_xlabel("operation index (millions)"); ax.set_ylabel("wires"); ax.set_ylim(830, peak + 5)
headline(fig, f"DQI: the decoder touches its full width ({peak} wires) again and again", f"static liveness of the portfolio DQI file, sampled every 20,000 operations; {v['dqi']['num_qubits']} qubits declared")
plt.show()

### 5.4 The rare failures (why the public score of the `cd71c482` line was a coin flip)
Every file we measured of the `cd71c482` line, including `bb4287e7` and our version-1 transplant, fails on roughly
6e-6 of the verifier's shots (an approximate rare branch inherited from the source). The board draws 102,400 shots per
file from the file's hash and a secret salt, so about half of the files of this line draw zero failures (R = 1.1913)
and half draw one or more (R = 1.1967 for one failure, about -0.04 on the score). Measured with the official verifier:

| file | default salt | `s1` | `s2` |
|---|---|---|---|
| `bb4287e7` | 0 | 0 | 1 |
| our version-1 transplant | 1 | 2 | 0 |
| `4c81b726` (the current portfolio DQI, maoxin123's rewrite of `cd71c482`) | 0 | 0 | 0 |

The portfolio DQI file `4c81b726` drew zero failures on all three salts (307,200 shots). That is encouraging but not a
proof: had it inherited the rate of ~6e-6, three clean draws would still happen with probability about 16%
(exp(-1.8)), and the private leaderboard draws again. We do not know whether the rewrite repaired the rare branch or
is lucky so far; **a decoder of this line that is exact on every shot would make the score salt-proof**: open problem 1
in section 8. The cell below prints the three runs from the solution notebook's receipt.

In [ ]:
for salt, r_ in DQI_SALTS.items(): print(f"4c81b726, salt {salt:8s}: {r_.get('status')}, Toffolis {r_.get('toffoli_count', math.nan):,.2f}, value errors {r_.get('value_errors')}, phase errors {r_.get('phase_errors')}, R {r_.get('repetition_factor', math.nan):.4f}")
print("shots verified without a failure:", 102400 * sum(1 for r_ in DQI_SALTS.values() if r_.get("value_errors") == 0 and r_.get("phase_errors") == 0))

### 5.5 Ablation: what each part of our version-1 change does (official verifier; history of an earlier portfolio)
Four builds on top of `bb4287e7`, each verified with the unchanged official DQI verifier (102,400 shots, default
salt): window region A only (`aligned_route`), region B only (`inverse12 products`), both (our version-1 transplant),
and both plus Heliosli's retained-product region, which depends on two other regions that `bb4287e7` rewrote.

In [ ]:
# Runs on Kaggle (five official DQI verifications).
import subprocess
kx = {k: to_kmx(p, TMP / f"dqi_{k}.kmx") for k, p in dqi_src.items() if k in ("cd71c482", "9fcda0c1", "bb4287e7")}
def scan_regions(path, keep):
    tags, kept, idx = [""], {}, 0
    for line in open(path):
        if line.startswith(("APPEND_TO_REGISTER", "REGISTER")): continue
        if line.startswith("INFO_TAG"): idx += 1; tags.append(line)
        if idx in keep: kept.setdefault(idx, []).append(line)
    return tags, kept
ALL = [673, 16783, 17409]
tA, RA = scan_regions(kx["cd71c482"], set(ALL)); _, RH = scan_regions(kx["9fcda0c1"], set(ALL)); tB, _ = scan_regions(kx["bb4287e7"], set())
sm = difflib.SequenceMatcher(a=tA, b=tB, autojunk=False)
amap = {blk.a + k: blk.b + k for blk in sm.get_matching_blocks() for k in range(blk.size)}
def build(regions, out):
    jm = {amap[i]: i for i in regions}; idx, done = 0, set()
    with open(out, "w") as f:
        for line in open(kx["bb4287e7"]):
            if line.startswith("INFO_TAG"): idx += 1
            if idx in jm and not line.startswith(("APPEND_TO_REGISTER", "REGISTER")):
                if idx not in done: f.writelines(RH[jm[idx]]); done.add(idx)
                continue
            f.write(line)
    kb = pathlib.Path(str(out).replace(".kmx", ".kmb")); km.Circuit.from_file(out, format="kmx").to_file(kb, format="kmb"); return kb
def verify_dqi(path, salt=""):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, str(VER / "dqi/verify_submission_dqi.py"), str(path), "--salt", salt], capture_output=True, text=True, env=env, cwd=str(VER), timeout=3600)
    return json.loads(r.stdout[r.stdout.index("{"):])
abl = {}
abl["bb4287e7 (base)"] = verify_dqi(dqi_src["bb4287e7"])
for name, regions in (("+ window region A", [673]), ("+ window region B", [16783]), ("+ both (our v1 transplant)", [673, 16783]), ("+ both + retained product", [673, 16783, 17409])):
    abl[name] = verify_dqi(build(regions, TMP / "abl.kmx")); log(name, abl[name]["toffoli_count"], abl[name]["value_errors"], abl[name]["phase_errors"])
names = list(abl); tof = [abl[n]["toffoli_count"] for n in names]; pe = [abl[n]["phase_errors"] for n in names]; ve = [abl[n]["value_errors"] for n in names]
print(f"{'build':28s} {'Toffolis/shot':>14s} {'vs base':>9s} {'value err':>10s} {'phase err':>10s}")
for n, t_, a_, b_ in zip(names, tof, ve, pe): print(f"{n:28s} {t_:14,.2f} {t_ - tof[0]:+9.2f} {a_:10d} {b_:10d}")
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.0))
cols = [MUTED] + [TC["dqi"]] * 3 + [C[7]]
axs[0].barh(names, [t_ - tof[0] for t_ in tof], color=cols, height=0.55); axs[0].invert_yaxis()
axs[0].set_title("Toffolis per shot vs bb4287e7", loc="left", fontsize=10); axs[0].axvline(0, color=INK2, lw=1)
axs[1].barh(names, pe, color=cols, height=0.55); axs[1].invert_yaxis(); axs[1].set_yticklabels([])
axs[1].set_title("phase errors in 102,400 shots", loc="left", fontsize=10)
for i, p_ in enumerate(pe): axs[1].text(p_, i, f" {p_}", va="center", color=INK)
headline(fig, "Each readout window saves 24 Toffolis on every shot; a lone transplant breaks the phase", "official DQI verifier, default salt")
plt.show()
print("""Reading: each window saves exactly 24 Toffolis on every shot and the two add up. The retained product saves one
more but needs two companion regions; alone it breaks the phase on ~1.7% of shots, which the verifier prices as a
3x repetition overhead. Cross-lineage transplants must move whole dependency groups, and must be verified.""")

### 5.6 How lucky is a board draw?
Pooling our official runs of the older files of this line (two files, three salts, 614,400 shots, plus the two
single-region builds above; `4c81b726` is not pooled) gives the failure rate; the board's 102,400-shot draw then has a Poisson number of failures, each of which moves
the repetition factor R and the score.

In [ ]:
# Runs on Kaggle (uses the ablation runs above; cross-checks our R formula against the verifier module). Illustration of the old rate applied to the current portfolio.
fails = [0, 0, 1, 1, 2, 0] + [abl[n]["value_errors"] + abl[n]["phase_errors"] for n in names[1:3]]
eps = sum(fails) / (102400 * len(fails)); lam = eps * 102400
sys.path.insert(0, str(VER / "dqi")); import verify_submission_dqi as vq
assert all(abs(vq.repetition_overhead(vq.clopper_pearson_upper_bound(k, 102400, -128.0)) - rep_factor(cp_upper(k))) < 1e-9 for k in range(5))
probs, scores = [], []
for k in range(5):
    pk = math.exp(-lam) * lam ** k / math.factorial(k)
    dq_ = rows["dqi"][2] / R0 * rep_factor(cp_upper(k))
    sc = (REF["invsqrt"] / rows["invsqrt"][2] * REF["dqi"] / dq_ * REF["dfthcsos"] / rows["dfthcsos"][2]) ** (1 / 3)
    probs.append(pk); scores.append(sc)
fig, ax = plt.subplots(figsize=(10.5, 3.0))
ax.bar([f"{k} failed" for k in range(5)], probs, color=TC["dqi"], width=0.55)
for k, (p_, s_) in enumerate(zip(probs, scores)): ax.text(k, p_, f"score {s_:.4f}", ha="center", va="bottom", color=INK, fontsize=9)
ax.set_ylabel("probability"); ax.set_ylim(0, max(probs) * 1.18)
headline(fig, f"A board draw of the older cd71c482-line files: P(zero failures) = {probs[0]:.2f}", f"Poisson with eps = {eps:.1e} per shot (pooled official runs); label = current portfolio score if its DQI drew that many")
plt.show()
print(f"expected failures per draw {lam:.2f}; P(0) = {probs[0]:.2f}; expected score {sum(p * s for p, s in zip(probs, scores)) / sum(probs):.5f}")

### 5.7 The DQI landscape: public files in qubits x Toffolis
Our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) indexes the public files too (scan of
2026-11-12); the points below come from manwithacat's dataset (scan of 2026-11-10), the black dot is the portfolio
file. Cost = R x Q<sup>2</sup> x T, so the grey lines of equal cost are T proportional to 1/Q<sup>2</sup>. Families sit on
vertical lines (one qubit count each); moving one qubit left is worth ~880 Toffolis.

In [ ]:
dq_pts = []
for r in man:
    if r["task"] != "dqi": continue
    c_ = fnum(r["dqi_zero_failure_cost"]) if r.get("dqi_zero_failure_cost") else fnum(r["local_task_score"]); q_ = fnum(r["qubits"])
    if c_ == c_ and q_ == q_ and c_ < 1e12: dq_pts.append((q_, c_ / (R0 * q_ * q_)))
D = np.array(dq_pts); D = D[(D[:, 0] >= 885) & (D[:, 0] <= 900)]
fig, ax = plt.subplots(figsize=(10.5, 3.8))
x0, x1 = D[:, 0].min() - 1, D[:, 0].max() + 1; y0, y1 = min(D[:, 1].min(), rows["dqi"][1]) * 0.985, np.percentile(D[:, 1], 85)
qq = np.linspace(x0, x1, 100)
for sc in (3.75e11, 3.9e11, 4.1e11, 4.4e11):
    ax.plot(qq, sc / (R0 * qq ** 2), color=RULE, lw=1)
    yr = sc / (R0 * x1 ** 2)
    if y0 < yr < y1: ax.text(x1, yr, f" {sc:.2e}", color=MUTED, fontsize=8, va="center")
    else:
        xa = math.sqrt(sc / (R0 * y1 * 0.97))
        if x0 < xa < x1: ax.text(xa, y1 * 0.97, f"{sc:.2e} ", color=MUTED, fontsize=8, ha="right")
ax.set_xlim(x0, x1 + 2.5); ax.set_ylim(y0, y1)
ax.scatter(D[:, 0], D[:, 1], s=26, color=TC["dqi"], edgecolor=SURF, linewidth=1.2, zorder=3, label="public file")
ax.scatter([rows["dqi"][0]], [rows["dqi"][1]], s=90, color=INK, zorder=4, label="portfolio")
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y / 1e3:,.0f}k"))
ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffolis per shot T"); ax.legend(loc="lower right", fontsize=8.5)
headline(fig, "DQI landscape: families sit on fixed qubit counts; the frontier is at 891-892 qubits", f"{len(D)} public files (manwithacat's dataset, costs at zero failed shots); grey = equal R Q^2 T")
plt.show()

<a id="s6"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">6 &middot; Chemistry deep dive</div>

**The block encoding** (paper Sec. III.C): a DF-THC factorization of the FeMoCo Hamiltonian (shape R, B, C) is loaded
row by row. Each row a of the factorization is a unit vector in 76 orbitals, written as 75 Givens angles of `brot`
bits on a fixed tree. The submitted loaders `ANGLE_k` stream these angles in batches of J words:
`|a>|z> -> |a>|z XOR delta_k(a)>` with `delta_k = batch_k XOR batch_(k-1)`, forward in U and backwards in U-dagger.
The state preparations and the Givens rotations themselves are not kickmix circuits; the verifier prices them with a
formula from (R, B, C) and `brot`. The score is lambda_eff x T x Q of the whole thing (section 2 shows the split).

### 6.1 The formula part: the shape (R, B, C) moves T x Q in steps
Only the ANGLE loaders are simulated. Below, the verifier's own cost model (`costs_prepare.py`) evaluated over the
basis size B at fixed R and C, and over R and C at fixed B, with the portfolio's loaders held fixed. Accuracy is not
modelled here: any other shape must still pass the 20 mHa gates with its own refit.

In [ ]:
sys.path.insert(0, str(VER / "dfthcsos" / "scoreinfra"))
from costs_prepare import SpinFreeData_V1 as Model
res = v["dfthcsos"]["resources"]
W_shape = np.load(TMP / "sub" / "dfthcsos" / "submission.npz")["W"].shape
print("factorization shape (R, B, C):", W_shape, " brot:", res["brot"], " J:", v["dfthcsos"].get("batch_size"), " loaders:", v["dfthcsos"].get("num_circuits"))
def model_cost(R, B, C, angles_T=res["angles_toffoli"], ang_pers=res["angles_persistent_qubits"], ang_tmp=res["angles_temporary_qubits"], beta=res["brot"]):
    m = Model(R=R, B=B, C=C, N=76, chi=10, beta=beta)
    o, i = m.outer_state_prep_qubit_cost(), m.inner_state_prep_qubit_cost()
    T = m.outer_state_prep() + m.inner_state_prep() + m.givens_rotations() + angles_T
    return T * (o["persistent"] + i["persistent"] + ang_pers + max(o["temporary"], i["temporary"], ang_tmp))
R0_, B0, C0 = W_shape; base_c = model_cost(R0_, B0, C0)
assert abs(base_c - v["dfthcsos"]["toffoli_count"] * v["dfthcsos"]["num_qubits"]) < 1e-6 * base_c
Bs = list(range(100, 200)); rel = []
for B in Bs:
    try: rel.append(model_cost(R0_, B, C0) / base_c)
    except ValueError: rel.append(np.nan)
rel = np.array(rel)
Rs, Cs = np.arange(4, 17), np.arange(30, 61, 2)
grid = np.full((len(Rs), len(Cs)), np.nan)
for i, R_ in enumerate(Rs):
    for j, C_ in enumerate(Cs):
        try: grid[i, j] = model_cost(int(R_), B0, int(C_)) / base_c
        except ValueError: pass
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axs[0]; ax.step(Bs, rel, where="mid", color=TC["dfthcsos"]); ax.axvline(B0, color=C[1], lw=1.2); ax.axhline(1, color=RULE, lw=1)
ax.text(B0, np.nanmax(rel), f" portfolio B = {B0}", color=INK2, va="top", fontsize=9)
ax.set_xlabel(f"basis size B (R = {R0_}, C = {C0}, loaders fixed)"); ax.set_ylabel("T x Q relative to the portfolio")
ax.set_title("over B", loc="left", fontsize=10)
ax = axs[1]; im = ax.imshow(grid, origin="lower", aspect="auto", cmap="Greens", extent=(Cs[0] - 1, Cs[-1] + 1, Rs[0] - 0.5, Rs[-1] + 0.5))
ax.scatter([C0], [R0_], s=60, color=C[1], edgecolor=SURF, linewidth=1.5, zorder=3); ax.annotate("portfolio", (C0, R0_), (C0 + 3, R0_ + 2.5), fontsize=8.5, color=INK, arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_xlabel("C (columns)"); ax.set_ylabel("R (ranks)"); ax.grid(False); cb = plt.colorbar(im, ax=ax, pad=0.02); cb.set_label("T x Q relative")
ax.set_title(f"over R and C (B = {B0})", loc="left", fontsize=10)
nxt = rel[Bs.index(B0) + 1] if B0 + 1 in Bs else np.nan
headline(fig, "Chemistry formula: T x Q moves in steps as the factorization shape changes",
         f"verifier's SpinFreeData_V1 model, loaders held fixed; B = {B0 + 1} would cost {100 * (nxt - 1):+.1f}%; accuracy (20 mHa gates) not modelled")
plt.show()
print(f"over B in [100, 199]: min {np.nanmin(rel):.3f}, max {np.nanmax(rel):.3f} of the portfolio's T x Q; "
      f"over R in [4, 16], C in [30, 60]: min {np.nanmin(grid):.3f}, max {np.nanmax(grid):.3f}")
print("""Reading: the formula part moves in steps (the lookup block sizes inside the formula change at thresholds such as B = 128),
and it moves with R and C too. Any cheaper shape must still pass the accuracy gates, which the portfolio already
presses against 20 mHa (section 1): finding one is a numerical factorization problem, not circuit surgery.""")

### 6.2 The loaders: cache monomials, two parities, one Toffoli per product
A loader must XOR, into 40 word bits (J = 5 words of `brot` = 8 bits in the portfolio), a fixed Boolean function of the
11 address bits. The public loaders (maoxin123's v22 line, building on Lera's, syed3000's, gengsr's and others' work,
section 9) do it in two phases, which the miniature below draws and the bars count:
1. **Cache**: `CCX` builds ~75 products of address bits (monomials) on scratch wires, one Toffoli each.
2. **Products**: free `CX` parities build two linear forms (of address bits and cached monomials) on two helper wires;
   one `CCX` multiplies them into a word bit, and free `CX` fan-outs copy it to other word bits that need the same term.

At the end the cache is cleared with `HMR` and classically controlled `CZ` (zero Toffolis, section 1). Every Toffoli
saved per pass is worth 2 Toffolis of T (the loaders run forward and in reverse).

In [ ]:
# Figure: a loader in miniature (schematic) and the Toffoli split of the portfolio's 15 loaders (counted in the files).
fig, axs = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1.25, 1]})
WL = ["a_i", "a_j", "cache", "f1", "f2", "w", "w'", "b:m"]
draw_circuit(axs[0], WL, [(0, "CCX", "a_i", "a_j", "cache"), (1, "CX", "a_i", "f1"), (2, "CX", "cache", "f1"), (3, "CX", "a_j", "f2"),
                          (4, "CCX", "f1", "f2", "w"), (5, "CX", "w", "w'"), (6, "BOX", "f1", "f2", {"label": "undo\nparities"}),
                          (7, "HMR", "cache", "b:m"), (8, "CZ", "a_i", "a_j", {"if": "b:m"})], 9)
axs[0].set_title("schematic: cache, parities, product, fan-out, free clean-up", loc="left", fontsize=10)
split = []
for p in CHEM_KMX:
    names_, regq = {}, collections.defaultdict(set)
    for line in open(p):
        t = line.split()
        if not t: continue
        if t[0] == "REGISTER": names_[t[1]] = t[2].strip('"')
        elif t[0] == "APPEND_TO_REGISTER": regq[t[2]].add(int(t[1][1:]))
    words = set().union(*[regq[r] for r, nm in names_.items() if nm.startswith("word")])
    o_, _ = parse(p); prod = sum(1 for o in o_ if o[0] == "CCX" and o[1][2] in words); allt = sum(1 for o in o_ if o[0] in TOFF)
    split.append((allt - prod, prod))
S = np.array(split); ks = np.arange(len(S)); ax = axs[1]
ax.bar(ks, S[:, 0], color=TC["dfthcsos"], width=0.75, label="cache (monomials)", edgecolor=SURF, linewidth=1.5)
ax.bar(ks, S[:, 1], bottom=S[:, 0], color=TC["dfthcsos"], alpha=0.45, width=0.75, label="products into word bits", edgecolor=SURF, linewidth=1.5)
for k in ks: ax.text(k, S[k].sum(), f"{S[k].sum()}", ha="center", va="bottom", fontsize=7, color=INK2)
ax.set_xticks(ks); ax.set_xlabel("loader k"); ax.set_ylabel("static Toffolis per pass")
ax.set_ylim(0, S.sum(axis=1).max() * 1.3); ax.legend(loc="upper right", ncol=2, fontsize=8.5); ax.grid(axis="x", visible=False)
ax.set_title(f"portfolio loaders: {S.sum():,} Toffolis per pass", loc="left", fontsize=10)
headline(fig, "Chemistry loaders: one Toffoli per product, after a shared cache of monomials",
         f"{S[:, 1].sum():,} product Toffolis and {S[:, 0].sum():,} cache Toffolis per pass; receipt: {res['angles_toffoli_per_pass']:,.0f} per pass, x2 passes")
plt.show()

### 6.3 The chemistry landscape: lambda_eff against T x Q
Each public chemistry submission is a point: the factorization sets lambda_eff (and, through the formula, part of T
and Q), the loaders set the rest. The gains came first from the factorization (new shapes and refits), then from the
loaders.

In [ ]:
ch_pts = [(fnum(r["lambda_eff"]), fnum(r["T_total"]) * fnum(r["Q_total"]), r["release_date"][:10]) for r in man if r["task"] == "chemistry"]
ch_pts = [p for p in ch_pts if p[0] == p[0] and p[1] == p[1]]
P = np.array([(a, b) for a, b, _ in ch_pts]); days = np.array([np.datetime64(d) for *_, d in ch_pts]).astype("datetime64[D]")
dnum = (days - days.min()).astype(float)
fig, ax = plt.subplots(figsize=(10.5, 3.9))
sc_ = ax.scatter(P[:, 1], P[:, 0], c=dnum, cmap="Greens", vmin=-dnum.max() * 0.3, s=26, edgecolor="#b9b8b3", linewidth=0.5, zorder=3)
ax.scatter([v["dfthcsos"]["toffoli_count"] * v["dfthcsos"]["num_qubits"]], [v["dfthcsos"]["lambda_eff"]], s=90, color=INK, zorder=4, label="portfolio")
TQ = np.geomspace(P[:, 1].min() * 0.9, P[:, 1].max() * 1.1, 50)
for sc in (3e8, 3.3e8, 3.7e8):
    ax.plot(TQ, sc / TQ, color=RULE, lw=1, zorder=1)
cb = plt.colorbar(sc_, ax=ax, pad=0.01); cb.set_label("release day"); cb.ax.set_ylim(0, dnum.max()); cb.set_ticks([0, dnum.max()]); cb.set_ticklabels([str(days.min()), str(days.max())])
ax.set_xscale("log"); ax.set_xlim(P[:, 1].min() * 0.97, np.percentile(P[:, 1], 95) * 1.05)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1e6:.1f}M")); ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, _: f"{x / 1e6:.1f}M")); ax.set_xlabel("T x Q of the block encoding"); ax.set_ylabel("lambda_eff (Ha)")
ax.set_ylim(P[:, 0].min() - 0.3, np.percentile(P[:, 0], 97) + 0.3); ax.legend(loc="upper right", fontsize=8.5)
headline(fig, "Chemistry landscape: lambda_eff fell first, then T x Q", f"{len(P)} public submissions (manwithacat's dataset); darker = later; grey = equal cost")
plt.show()

<a id="s7"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">7 &middot; The public lineage: how the frontier was built, and by whom</div>

This competition is a relay race in public. manwithacat's *VQCC public circuits* dataset records, for every public
circuit, its release time, its publisher and its cost under the official verifier. The best public ratio per task by
release time shows where the community found its gains; the labels name the publishers of the largest steps.

In [ ]:
# Figure: the best public circuit per task over time, with the largest steps labelled by publisher.
task_key = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}
series, steps = {}, {}
for t_, key in task_key.items():
    pts = sorted((r["release_date"][:16], fnum(r["dqi_zero_failure_cost"]) if t_ == "dqi" and r.get("dqi_zero_failure_cost") else fnum(r["local_task_score"]), r["author"]) for r in man if r["task"] == t_)
    best, out = math.inf, []
    for d, c, a in pts:
        if c == c and c < best: best = c; out.append((np.datetime64(d.replace(" ", "T")), REF[key] / c, a))
    series[t_] = out
fig, axs = plt.subplots(3, 1, figsize=(10.5, 8.2), sharex=True)
for ax, (t_, out) in zip(axs, series.items()):
    key = task_key[t_]; d, r, a = zip(*out); col = TC[key]
    ax.step(d, r, where="post", color=col); ax.scatter(d, r, s=16, color=col, zorder=3, edgecolor=SURF, linewidth=0.8)
    ax.axhline(ratios[key], color=INK, lw=1, ls=(0, (4, 3))); ax.text(d[0], ratios[key], f" portfolio {ratios[key]:.2f}x", va="bottom", fontsize=8.5, color=INK)
    jumps = sorted(range(1, len(r)), key=lambda i: r[i] / r[i - 1], reverse=True)[:4]
    for n_, i in enumerate(sorted(jumps)):
        ax.annotate(f"{a[i].split(' (')[0][:22]}\n+{100 * (r[i] / r[i - 1] - 1):.1f}%", (d[i], r[i]), (0, -22 - 24 * (n_ % 2)), textcoords="offset points", ha="center", va="top", fontsize=7.5, color=INK2,
                    arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
    lo, hi = min(r) * 0.92, max(max(r), ratios[key]) * 1.04
    ax.set_ylim(lo - (hi - lo) * 0.6, hi); ax.set_ylabel("ref / best cost"); ax.set_title(f"{TN[key]}: {len(out)} frontier steps", loc="left", fontsize=10, color=INK)
fig.autofmt_xdate()
headline(fig, "The public frontier, step by step: many small gains, a few big jumps",
         "best public ratio per task by release time (manwithacat's dataset); labels = publisher of the four largest steps; dashed = this portfolio")
plt.show()
for t_, out in series.items(): print(f"{t_:9s} {len(out):3d} frontier records; last at {str(out[-1][0])[:16]}: {out[-1][1]:.3f}x by {out[-1][2]}")

### The DQI relay in the last days
The dataset scan behind the timeline ends on 2026-11-10, so the newest steps are not in the figure above (the
[QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) scan of 2026-11-12 lists them).
Three public bests, in order: **Atakan Aldemir**'s `eae396ba` (four-Toffoli GF(64) inverse character on `bb4287e7`;
394,089 Toffolis per shot, 892 qubits, cost 3.735e11); **Areeb Ali**'s `8eccde65` (`eae396ba` plus 133 retained products
in idle qubits, public best score 25.40767 in a full portfolio); and **maoxin123**'s `4c81b726` (a gate-level rewrite of
`cd71c482` in three waves, **not** derived from `eae396ba`: algebraic pull-backs of repeated bilinear and quadratic
forms and clean-target gadgets in `unextract`, then two waves of content-matched reductions; 392,016 Toffolis per shot,
892 qubits, cost 3.716e11), about 2,070 Toffolis below the previous best decoder. Our solution notebook joins it with
the best invsqrt and chemistry: score 25.44954 (local, default salt).

### Who moved the frontier
Counting, per publisher, the public files that set a new best for a task. It is a crude measure (one big idea and
one small re-tune both count as one, and a republication is credited to its publisher), but it shows how many people
the frontier depends on. Section 9 credits the ideas, not just the files.

In [ ]:
cnt = collections.defaultdict(collections.Counter)
for t_, out in series.items():
    for d, r, a in out: cnt[re.sub(r"^team ", "", re.sub(r"\s*\(.*\)$", "", a).strip())][task_key[t_]] += 1
people = sorted(cnt, key=lambda p: -sum(cnt[p].values()))
topn = people[:20]
fig, ax = plt.subplots(figsize=(10.5, 5.0))
left = np.zeros(len(topn))
for key in ("invsqrt", "dqi", "dfthcsos"):
    vals = np.array([cnt[p][key] for p in topn]); ax.barh(range(len(topn)), vals, left=left, color=TC[key], height=0.66, label=TN[key], edgecolor=SURF, linewidth=1.5); left += vals
for i, tot in enumerate(left): ax.text(tot + 0.2, i, f"{int(tot)}", va="center", fontsize=8.5, color=INK2)
ax.set_yticks(range(len(topn))); ax.set_yticklabels(topn); ax.invert_yaxis(); ax.legend(loc="lower right", fontsize=9); ax.grid(axis="y", visible=False)
ax.set_xlabel("public files that set a new best for a task")
headline(fig, f"{len(people)} publishers set at least one public best; the top 20 are shown",
         "frontier records from manwithacat's public-circuits dataset, by publisher and task")
plt.show()

### The family tree of the portfolio
A simplified map of what each portfolio file is built on. The DQI row now shows two branches: maoxin123's rewrite of `cd71c482` (the portfolio file) and the older
Atakan / Areeb branch (earlier public bests; `8eccde65` is `eae396ba` plus 133 retained products), which `4c81b726` is not derived from, as credited by its authors (section 9 has every name and
link). Arrows read *is built on / contributes to*.

In [ ]:
# Figure: the portfolio's family tree (curated from the authors' own credits; simplified).
nodes = {  # key: (x, y, task, label)
    "i0": (0, 4.4, "invsqrt", "62-qubit line: Syed Asad Ali,\nHosen42, Kirill Tymchishin,\nyoyobar, busyaprime, mrmeows, ..."),
    "i1": (1, 4.95, "invsqrt", "maoxin123 a15b0d73\nstage re-order, CSWAP rewrites"),
    "i2": (1, 3.85, "invsqrt", "Gengsr 26ebf39d\n46-CSWAP shifter"),
    "i3": (2, 4.4, "invsqrt", "manwithacat HW-25\n6c6a734d (join)"),
    "i4": (2, 5.35, "invsqrt", "WCao24 94b29cc5\n44-CSWAP normaliser"),
    "i5": (2, 3.45, "invsqrt", "Thomas Gamet 5ec4f516\nearly prefix"),
    "i6": (3.2, 4.4, "invsqrt", "PORTFOLIO invsqrt\n9c708f7b: our splice\nat clean cuts"),
    "d0": (0, 2.2, "dqi", "decoder line: jday96314,\nSyed Asad Ali, SomeJoint,\nKirill Tymchishin, ..."),
    "d1": (1, 2.2, "dqi", "team i want to run gemma 9b\ncd71c482 (zero-chart\nrewrites)"),
    "d2": (2, 2.2, "dqi", "Atakan Aldemir bb4287e7\nshared products"),
    "d3": (2, 1.25, "dqi", "Atakan Aldemir e9f54be3\nreadout windows (regions\nvia Heliosli 9fcda0c1)"),
    "d4": (3.2, 2.2, "dqi", "our v1 transplant\nreadout windows\nonto bb4287e7 (-48)"),
    "d5": (4.3, 2.2, "dqi", "Atakan Aldemir eae396ba\nGF(64) character (our\nearlier portfolio DQI)"),
    "d6": (3.2, 3.0, "dqi", "PORTFOLIO DQI\n4c81b726: maoxin123\n392k rewrite of cd71c482"),
    "d7": (5.4, 2.2, "dqi", "Areeb Ali 8eccde65\nproduct retention\n(previous public best)"),
    "c0": (0, 0.0, "dfthcsos", "Lera (marculera)\n(8, 143, 44) table\nand angle words"),
    "c1": (1, 0.0, "dfthcsos", "ozertuu table 8afd4c67\n+ cache dataset"),
    "c2": (2, 0.0, "dfthcsos", "prvsiyan\nconstrained refit"),
    "c3": (3.2, 0.0, "dfthcsos", "Heliosli\nweight update e9cffb84"),
    "c4": (3.2, -0.95, "dfthcsos", "maoxin123 v22 loaders\n4,089 Toffolis per pass"),
    "c5": (4.3, -0.45, "dfthcsos", "PORTFOLIO chemistry"),
}
edges = [("i0", "i1"), ("i0", "i2"), ("i1", "i3"), ("i2", "i3"), ("i3", "i6"), ("i4", "i6"), ("i5", "i6"),
         ("d0", "d1"), ("d1", "d2"), ("d2", "d4"), ("d3", "d4"), ("d4", "d5"), ("d5", "d7"), ("d1", "d6"),
         ("c0", "c1"), ("c1", "c2"), ("c2", "c3"), ("c3", "c5"), ("c4", "c5")]
nodes = {k_: (x * 1.25, y_, t_, l_) for k_, (x, y_, t_, l_) in nodes.items()}
fig, ax = plt.subplots(figsize=(11, 6.6)); ax.set_xlim(-0.65, 7.35); ax.set_ylim(-1.5, 5.9); ax.axis("off")
bw, bh = 1.08, 0.66
RAD = {("d1", "d6"): -0.2}
for a_, b_ in edges:
    (x1, y1, *_), (x2, y2, *_) = nodes[a_], nodes[b_]
    ax.annotate("", (x2 - bw / 2, y2), (x1 + bw / 2, y1), arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=1, shrinkA=0, shrinkB=2, connectionstyle="arc3,rad=" + str(RAD.get((a_, b_), 0.0))))
for k_, (x, y_, task, lab) in nodes.items():
    pf = lab.startswith("PORTFOLIO"); col = TC[task]
    ax.add_patch(FancyBboxPatch((x - bw / 2, y_ - bh / 2), bw, bh, boxstyle="round,pad=0.02", facecolor=col if pf else SURF, edgecolor=col, lw=2 if pf else 1.2))
    ax.text(x, y_, lab, ha="center", va="center", fontsize=7.3, color="white" if pf else INK, fontweight="bold" if pf else "normal")
for task, y_ in (("invsqrt", 5.75), ("dqi", 2.95), ("dfthcsos", 0.75)): ax.text(-0.6, y_, TN[task], fontsize=11, fontweight="bold", color=TC[task])
headline(fig, "The portfolio's family tree: three relay races, dozens of authors",
         "simplified, from the authors' own credit notes; filled boxes = the files in this portfolio")
plt.show()

<a id="s8"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">8 &middot; What does not work, and open problems</div>

Negative results save everyone time, so here are ours, with the measurement behind each. If you have made one of
them work, please say so in the comments: we will correct this table and credit you.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #fdeceb;"><th style="text-align: left; padding: 5px 8px;">Task</th><th style="text-align: left; padding: 5px 8px;">Idea</th><th style="text-align: left; padding: 5px 8px;">What happened</th><th style="text-align: left; padding: 5px 8px;">Where</th></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="text-align: left; padding: 5px 8px;">Merge the two phase streams (pay the 8 shared rotation weights once)</td><td style="text-align: left; padding: 5px 8px;">The stream-1 source bits are overwritten before stream 2 exists. Keeping them costs 8 qubits (~13% of the cost) to save at most 8 rotations (~57 Toffoli-eq., ~4%).</td><td style="text-align: left; padding: 5px 8px;">4, two phase streams</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="text-align: left; padding: 5px 8px;">Trust a splice found on a small input sample</td><td style="text-align: left; padding: 5px 8px;">On 512 shots the search found a splice 1.2 Toffolis cheaper; on 4,098 shots the gain vanished. Conditional blocks fire on random outcomes: re-measure every candidate on a large fresh sample.</td><td style="text-align: left; padding: 5px 8px;">4, splicing</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="text-align: left; padding: 5px 8px;">Transplant Heliosli's retained-product region alone</td><td style="text-align: left; padding: 5px 8px;">Saves one Toffoli per shot but breaks the phase on ~1.7% of shots (R ~3x): it depends on two companion regions that this line rewrote.</td><td style="text-align: left; padding: 5px 8px;">5.5, ablation</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="text-align: left; padding: 5px 8px;">More cross-lineage transplants (50 public files scanned region by region; earlier portfolio)</td><td style="text-align: left; padding: 5px 8px;">The only remaining candidates (one Toffoli each) depend on rewritten regions and fail the official verifier when moved alone.</td><td style="text-align: left; padding: 5px 8px;">our solution notebook</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="text-align: left; padding: 5px 8px;">Remove one qubit with a local patch</td><td style="text-align: left; padding: 5px 8px;">The decoder reaches its full width in hundreds of short episodes inside the GF(2<sup>12</sup>) multipliers; one qubit less needs a narrower multiplier everywhere.</td><td style="text-align: left; padding: 5px 8px;">5.3, liveness</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #1baf7a;">chemistry</b></td><td style="text-align: left; padding: 5px 8px;">Re-plan the loaders to beat 4,089 Toffolis per pass</td><td style="text-align: left; padding: 5px 8px;">No loader improved (two planner runs, 2026-11-12). Our parameter count (an estimate) puts maoxin123&rsquo;s architecture (products in five 3-dimensional blocks) at ~210 products for a generic loader against 212 used for loaders 1&ndash;3 (193 for loader 4): it is at its counting floor. With full mixing the floor is ~25 products lower (~186 against 212; ~165 against 193 for loader 4), but a block matcher with cross-block pairs reached 229 and greedy constructions 240&ndash;250, both above 193; removing any one product or any pair of products leaves the target uncovered, so flip-graph and local search find no move.</td><td style="text-align: left; padding: 5px 8px;">6.2 (our planner runs)</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="text-align: left; padding: 5px 8px;">Drop, snap or merge rotations of the 41 phase rotations to save Toffoli-equivalents</td><td style="text-align: left; padding: 5px 8px;">The angles are already max-margin optimal: re-solving the linear program for all 41 gains 2 units of a 58.6M-unit margin (the worst input uses 99.93% of the tolerance). Dropping any one rotation and re-tuning the other 40 is infeasible (best case, the smallest weight 2<sup>-35</sup> half-turn = 266M units, leaves a margin of &minus;48.5M; break-even needed +23.5M). Every weight exceeds twice the remaining margin, no weight can snap to T, and no pair of rotations fires on the same inputs (closest pairs agree on 99.99% of x, not 100%). More gain needs a better approximation, not rotation farming.</td><td style="text-align: left; padding: 5px 8px;">4, error budget (our run, 2026-11-12)</td></tr>
<tr><td style="padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="text-align: left; padding: 5px 8px;">Go from 892 to 891 qubits on our DQI file (the 891-qubit line of <code>fc3ba816</code>)</td><td style="text-align: left; padding: 5px 8px;">The 891 line saves its qubit by retaining one fewer AND product across the multiplier peaks and pays ~+2.8k Toffolis elsewhere, so it is not a smarter multiplier. Our 892-qubit file (<code>eae396ba</code>, same family as <code>8eccde65</code>) touches 892 live wires in 635 places; splitting a retained product (+1 Toffoli each) covered only 48 of them with 23 splits (587 uncovered); a later scan found an AND-split candidate at both ends of 199 of the 635 peaks. No 891-qubit file was built. One qubit is worth ~0.22% of the DQI cost, i.e. ~880 Toffolis.</td><td style="text-align: left; padding: 5px 8px;">5.3, liveness (our run, 2026-11-12)</td></tr>
</table>

### Open problems (ideas welcome in the comments)
1. **An exact DQI on the `cd71c482` line.** The older files fail on ~6e-6 of shots; `4c81b726` (a rewrite of that line)
   drew zero failures at three salts, which is weak evidence (about 16% chance if it had inherited the rate). Find out
   whether it is exact, or repair the rare branch for fewer Toffolis than a failure costs (~1,800 Toffolis per failed
   shot in a draw, section 2): the score becomes salt-proof.
2. **A single phase stream for invsqrt.** Fewer rotations fit the error margin (our linear-program estimate over all x:
   33 instead of 41), but not by re-tuning the 41 angles (table above); the design must keep the table's low bits alive or seed the product's accumulator with them.
3. **A narrower GF(2<sup>12</sup>) multiplier for DQI.** One qubit less is worth ~880 Toffolis (-0.22% of the DQI cost);
   our retained-product splitting covers only part of the 892-wire peaks (table above).
4. **A cheaper chemistry shape within the gates.** The formula part moves by -10% to +50% over nearby (R, B, C)
   (section 6.1); each candidate needs its own factorization and refit to pass 20 mHa.
5. **Fully mixed chemistry products.** The counting argument leaves ~25 products per loader on the table, if a
   construction can find them; local search, flip-graph moves and a block matcher did not.
6. **More transplants as new files appear.** The region profile (5.2) compares two DQI files of the same structure in
   minutes; newer files may again miss an older trick.

If you test one of these, please post what happened, negative results included. We will credit every contribution in
the next version.

<a id="s9"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">9 &middot; Full credits and licences</div>

The credits of the portfolio files (as in our solution notebook), then the credits of this guide. The public circuits
are indexed in our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) dataset.

**invsqrt** `9c708f7b`
- asteriospanoras (our solution notebook): the splice of the two circuits below at clean cuts, -3.0 Toffolis vs the best one.
- Thomas Gamet (with OpenAI Codex): the early-prefix circuit `5ec4f516`
  ([Quantum Attempt - new base bundle](https://www.kaggle.com/code/thomasgamet/quantum-attempt-new-base-bundle)).
- WCao24: the 44-CSWAP input normaliser `94b29cc5`, -2.0 Toffolis
  ([VQCC Deterministic Open Portfolio](https://www.kaggle.com/code/wcao24/vqcc-deterministic-open-portfolio)).
- maoxin123: `a15b0d73`, Block A shift-and-add stages re-ordered, controlled-SWAP rewrites
  ([LB 25.303 | Combined invsqrt + refit chem](https://www.kaggle.com/code/maoxin123/lb-25-303-combined-invsqrt-refit-chem)).
- manwithacat (James Barlow): HW-25 join and the public circuits dataset
  ([VQCC Hedge Wizardry](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)); Gengsr: the 46-CSWAP shifter
  ([LB 25.2268](https://www.kaggle.com/code/gengsr/lb-25-2268-shifter-46-own-chemistry-loaders)).
- The 62-qubit lineage: Syed Asad Ali (base circuit), Hosen42 (q25 CCX merge, measure-first rewrites), Kirill
  Tymchishin (carry port), Atakan Aldemir (CSWAP deletion), Ertuğrul Özer (`13b5bf7a` composition), busyaprime (Akmal
  Xodarev), yoyobar (19-bit conditional phase fusion), mrmeows / team selenium (the 63-qubit ancestor), Leo Tang and
  Ehan Mallick, as credited in their notebooks.

**DQI** `4c81b726` (maoxin123's 392k-Toffoli decoder, the portfolio's DQI)
- maoxin123: the decoder `4c81b726`, a gate-level rewrite of `cd71c482` in three waves (algebraic pull-backs of
  repeated bilinear and quadratic forms, clean-target gadgets in `unextract`, two waves of content-matched
  reductions), 392,016 mean Toffolis at 892 qubits, 0 failures at the default salt, `s1` and `s2`
  ([QCC v22 | a 392k-Toffoli DQI decoder](https://www.kaggle.com/code/maoxin123/qcc-v22-a-392k-toffoli-dqi-decoder),
  dataset [qcc-dqi-392k-toffoli-decoder](https://www.kaggle.com/datasets/maoxin123/qcc-dqi-392k-toffoli-decoder), CC0).
- Its base `cd71c482`: the team *i want to run gemma 9b*, Oleg Sher (sheroleg), Heliosli, Syed Asad Ali, SomeJoint and
  jday96314; the decoder line goes back to jday96314's continued-fraction decoder (its repository states no licence, so
  it is credited, not relicensed), SomeJoint's kit and narrow return, Ali Haydar Özdağ's flag carry and Syed Asad Ali's C1
  line, with product reuse by Atakan Aldemir, MAIDANG/daifanhao, Isabe11aM00re and Heliosli, and Kirill Tymchishin's
  measurement identity (as credited by maoxin123).
- Areeb Ali: product retention `8eccde65` (`eae396ba` plus 133 retained products in idle qubits), the previous public
  best ([LB 25.40767](https://www.kaggle.com/code/areebbb/product-retention-133-free-toffolis-lb-25-40767)).

**DQI history: the files of earlier versions of this guide** (`eae396ba` on `bb4287e7`)
- Atakan Aldemir: four-Toffoli GF(64) inverse character at 28 sites, -27.6 Toffolis per shot, `eae396ba`
  ([Quantum Circuits: Iso-Gap + DQI h5](https://www.kaggle.com/code/atakanaldemir/quantum-circuits-iso-gap-dqi-h5));
  earlier the five-Toffoli version and `bb4287e7`
  ([Shared Products and Controlled Refits](https://www.kaggle.com/code/atakanaldemir/qcc-shared-products-and-controlled-refits)),
  and the controlled-swap readout windows of `e9f54be3`
  ([Terminal Readout and Phase Folds](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)).
- asteriospanoras (our solution notebook, version 1): the readout windows transplanted onto `bb4287e7`, -48 Toffolis per shot.
- Team *i want to run gemma 9b*: `cd71c482` and `b4dc000c` (joint zero-chart rewrites, conditional reverse-phase
  schedule, prefix fusion), shipped publicly via Oleg Sher's notebook.
- Heliosli (`9fcda0c1`: retained products; the regions we transplanted were taken from this file), Kirill Tymchishin
  (measurement-order identities, retained-product audit), Syed Asad Ali (decoder `1ab3ac53`, measurement peephole),
  SomeJoint (narrow return), Ali Haydar Özdağ (flag-carry idea), Isabe11aM00re, MAIDANG, busyaprime and yoyobar (product
  caching), Leo Tang, mrmeows, manwithacat (release-and-recompute), daifanhao, Khattar and Gidney (2024, the paper the
  decoder follows), and jday96314 (the decoder design).

**Chemistry**
- maoxin123: v22 loaders, **4,089 Toffolis per pass** (five-block low-monomial subspace search), used from version 5
  ([QCC v22 | 4089-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4089-toffoli-chemistry-loaders)).
- Heliosli: weight update `e9cffb84` of prvsiyan's weights with prvsiyan's refit code, and the
  [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live) dataset this version reads
  ([The dataset is public now](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)).
- Lera (marculera): the (8, 143, 44) DF-THC table and its angle words
  ([dfthc-c44-build](https://www.kaggle.com/datasets/marculera/dfthc-c44-build)), and the authors her NOTICE credits
  (Ahmed Berat Özer, kozykappa, Kozykorpesh and Sumit Chongder, mrmeows, michaelrustad, phucthaiv02, somejoint,
  gengsr, qwertykeypadapple, lightningv08, syed3000).
- prvsiyan: constrained refit of the weights `f21b8a87` (used up to version 4)
  ([Verifier-Guided Quantum Circuit Optimization](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization)).
- maoxin123: v22 loaders, 4,097 Toffolis per pass (used up to version 4)
  ([QCC v22 | 4097-Toffoli chemistry loaders](https://www.kaggle.com/code/maoxin123/qcc-v22-4097-toffoli-chemistry-loaders)).
- ozertuu (Ertuğrul Özer): table `8afd4c67` and the cache dataset. Syed Asad Ali: the 8 x 143 x 44 table and the
  loader compiler. Lera (marculera): the DF-THC pipeline and the C44 build. Ahmed Berat Özer: progressive rounding and
  group compression. Atakan Aldemir: the pi-gauge. Weight refits before prvsiyan's: Isabe11aM00re, maoxin123 (with
  manwithacat's `dfthc_refit`), Heliosli, busyaprime. Dmitrii Gluzdov, milos and Agniv Banerjee: earlier shapes.

**Assembly and verification practice**: Alex Chilton's [Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
(pinning by SHA-256, re-verifying with the stock verifiers, pricing a DQI file at its board draw), and WCao24's
deterministic, byte-reproducible build. **Analysis**: Oleg Sher's [Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained).

**This guide.** `kmsim`, the figures and the measurements: asteriospanoras. The public-circuit landscapes, the timeline
and the publisher counts are drawn from manwithacat's [VQCC public circuits](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits)
dataset (Apache-2.0). The task descriptions follow the competition pages and the hosts' paper
([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)); the verifier formulas are those of the official verifiers;
the two-stream reading of invsqrt follows Oleg Sher's notebook; the chemistry loader description draws on maoxin123's
and Lera's published notes. Page design patterns follow common Kaggle notebook HTML practice.

Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each circuit keeps its authors' terms; nothing here is
relicensed. The code of this notebook (including `kmsim`) is Apache-2.0.

<div style="background: linear-gradient(120deg, #0b1f4b 0%, #2a78d6 70%, #1baf7a 100%); color: #ffffff; padding: 14px 18px; border-radius: 10px; margin-top: 8px;">
<b>Thank you for reading.</b> If this guide helped you, an upvote helps others find it; corrections, negative results
and ideas in the comments make the next version better.
</div>

In [ ]:
log("done")